# OOA model with nanochat on **Kaggle**: pretrain, then train on our use case

Same pipeline as the Colab notebook. Set these in the right-hand panel first:
**Accelerator -> GPU T4 x2 (or P100)**, **Internet -> On**, and **Persistence** as you like.

Kaggle keeps files in `/kaggle/working` only for the session. To keep your work:
1. At the end run the last cell, then **Save Version** (Save & Run All, or Quick Save). Download `ooa_model_bundle.zip` from the Output tab.
2. To continue later, upload that zip as a **Kaggle Dataset** (or add this notebook's output as an input). Cell 2 finds the
   `nanochat_base` folder in `/kaggle/input` and restores it, so training resumes.

**If you already pretrained** (model files from Colab): put them in a dataset, then run cells 1, 2, then skip to cell 6.
Skip cells 3-5 (data download, tokenizer and pretraining), they are only for pretraining from zero.
Free Kaggle gives about 30 GPU hours a week; one session can run up to 12 hours.

In [ ]:
#@title 1. Config
PROFILE = "t4"             # "t4" = real free-Colab run, "smoke" = quick pipeline check (tiny model), "a100" = Colab Pro (A100/L4)
DRIVE_DIR = "/kaggle/working/ooa-nanochat"   # checkpoints, data and tokenizer live here
SHARED_FOLDER_ID = ""      # optional: Drive ID of the ooa-nanochat folder shared from another Google account (from its link)
NANOCHAT_COMMIT = "92d63d4e8bb4df75c3b71618f31ddde2378b2bcd"   # version this notebook was written against
PLATFORM_REPO = "https://github.com/OneOfficeautomation-Private-Limited/OneOfficeAutomation-Platform"
VOCAB_SIZE = 16384         # tokenizer size (nanochat default is 32768; smaller suits a small model)

PROFILES = {
  #           depth, seq, device_batch, total_batch_tokens, target_tokens, data_shards, tok_chars
  "smoke": dict(depth=4,  seq=512,  dbs=16, total_batch=16384,  tokens=5_000_000,    shards=2, tok_chars=100_000_000),
  "t4":    dict(depth=8,  seq=1024, dbs=8,  total_batch=65536,  tokens=300_000_000,  shards=8, tok_chars=500_000_000),
  "a100":  dict(depth=12, seq=2048, dbs=16, total_batch=262144, tokens=1_000_000_000,shards=16,tok_chars=1_000_000_000),
}
P = PROFILES[PROFILE]
NUM_ITERS = P["tokens"] // P["total_batch"]

# SFT (training on our use case)
SFT_SEQ = 2048             # flows longer than this get cut; the T4 cannot afford much more
SFT_DEVICE_BATCH = 4
SFT_EPOCHS = 2 if PROFILE == "smoke" else 6      # passes over the OOA data (the seed set is small)
FLOW_REPEAT = 5            # real flows repeated per pass; was 15, which made the model recite whole training flows (memorising)
SYNTH_TRAIN = 1500 if PROFILE == "smoke" else 30000   # generated flows (no LLM), see gen_flows.py
SYNTH_EVAL = 20            # unseen generated flows scored by exact action sequence
SYNTH_PATTERN = 2000 if PROFILE == "smoke" else 12000   # long, multi-toolkit flows following real-flow patterns (gen_flows.real_patterns)
# Reinforcement layer (after SFT)
RL_PROMPTS = 60 if PROFILE == "smoke" else 400     # requests for RL; each gets RL_SAMPLES answers scored by the verifier
RL_SAMPLES = 8
RL_EXAMPLES_PER_STEP = 8
RL_EVERY = 20 if PROFILE == "smoke" else 25        # evaluate and save every N RL steps
SELF_IMPROVE_ROUNDS = 2
SI_PROMPTS = 20 if PROFILE == "smoke" else 100     # fresh requests per self-improvement round
SI_SAMPLES = 8
EVAL_SOURCE = "sft"        # which model the test cell loads: "sft" or "rl"
SFT_ITERS = 400 if PROFILE == "smoke" else 8000            # counted in MICRO-batches (4 per optimizer step): 8000 = 2000 steps, about 35 min on a T4
SMOLTALK_ROWS = 3000       # a little general chat data mixed in so the model still talks; 0 = OOA data only
MAX_FLOW_CHARS = 9000      # skip training flows longer than this (about SFT_SEQ tokens)
print(PROFILE, P, "iterations:", NUM_ITERS)
print("SFT settings -> SYNTH_TRAIN:", SYNTH_TRAIN, "| SFT_EPOCHS:", SFT_EPOCHS, "| SFT_ITERS cap:", SFT_ITERS)

In [ ]:
#@title 2. Setup (Kaggle): GPU, internet, nanochat, dependencies
import os, sys, shutil, glob, urllib.request, torch
assert torch.cuda.is_available(), "No GPU. Right panel: Settings -> Accelerator -> GPU T4 x2 (or P100)"
cap = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(0), "compute capability", cap, "| GPUs:", torch.cuda.device_count(), "(this notebook uses one)")
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise RuntimeError("No internet. Right panel: Settings -> Internet -> On (Kaggle asks for phone verification once).")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["NANOCHAT_BASE_DIR"] = DRIVE_DIR + "/nanochat_base"
# pre-Ampere GPUs (T4, P100) have no bf16: use fp16 with loss scaling. If you see NaN losses, set "float32" (slower).
os.environ["NANOCHAT_DTYPE"] = "bfloat16" if cap >= (8, 0) else "float16"
os.environ["OMP_NUM_THREADS"] = "1"
print("compute dtype:", os.environ["NANOCHAT_DTYPE"])

# restore an earlier model that you added as a Kaggle dataset (see the note at the top)
base = os.environ["NANOCHAT_BASE_DIR"]
for src in glob.glob("/kaggle/input/*/nanochat_base") + glob.glob("/kaggle/input/*/*/nanochat_base"):
    if not os.path.exists(base):
        shutil.copytree(src, base)
        print("restored model files from", src)

if not os.path.exists("/kaggle/working/nanochat"):
    !git clone -q https://github.com/karpathy/nanochat /kaggle/working/nanochat
!cd /kaggle/working/nanochat && git checkout -q {NANOCHAT_COMMIT}
# Keep Kaggle's own torch; install only nanochat's other dependencies
!pip -q install rustbpe tiktoken kernels filelock pyarrow psutil requests wandb jsonschema
%cd /kaggle/working/nanochat

## Pretraining (general language, on nanochat's web-text dataset)

In [ ]:
#@title 3. Download pretraining data and train the tokenizer (skipped if already done)
tok_pkl = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "tokenizer", "tokenizer.pkl")
!python -m nanochat.dataset -n {P["shards"]}
if not os.path.exists(tok_pkl):
    !python -m scripts.tok_train --max-chars={P["tok_chars"]} --vocab-size={VOCAB_SIZE}
    !python -m scripts.tok_eval
else:
    print("tokenizer already trained:", tok_pkl)

In [ ]:
#@title 4. Pretrain (resumes automatically after a disconnect)
import glob, re
ckpt_dir = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "base_checkpoints", f"d{P['depth']}")
steps = sorted(int(re.search(r"model_(\d+)\.pt", f).group(1)) for f in glob.glob(ckpt_dir + "/model_*.pt"))
resume = f"--resume-from-step={steps[-1]}" if steps else ""
if steps and steps[-1] >= NUM_ITERS:
    print("Pretraining already finished at step", steps[-1])
else:
    print("resuming from", steps[-1] if steps else "scratch")
    !python -m scripts.base_train --depth={P["depth"]} --head-dim=64 --window-pattern=L \
        --max-seq-len={P["seq"]} --device-batch-size={P["dbs"]} --total-batch-size={P["total_batch"]} \
        --num-iterations={NUM_ITERS} --eval-every=500 --eval-tokens={P["dbs"]*P["seq"]*20} \
        --core-metric-every=-1 --sample-every=1000 --save-every=250 --run=dummy {resume}

In [ ]:
#@title 5. Quick look at the pretrained model (it continues text; it does not follow instructions yet)
!python -m scripts.base_eval --eval=bpb,sample --device-batch-size=1 --split-tokens=65536 2>&1 | tail -25

## Training on our use case (OOA flows)

Builds training data from the platform repo with no LLM: requests to Flow JSON from the gold flows (3 wordings each),
plus catalog questions ("which action does X", "what inputs does Y take"). Some workflow families are **held out** and
never trained on; they are used in the final test. This seed set is small (about 60 flow examples): replace
`train.jsonl` with the full dialogue set when you have it.

In [ ]:
#@title 6. Build OOA training data
import json
os.makedirs("/kaggle/working/ooa", exist_ok=True)
FILES = {"build_catalog.py": "\"\"\"Build the OOA action catalog from the platform's generated toolkit reference pages.\n\nUsage:\n    python build_catalog.py PLATFORM_REPO_DIR [OUT_JSONL]\n\nReads PLATFORM_REPO_DIR/docs/reference/toolkits/*.md (generated from `ooa manifest`) and writes one JSON object\nper action / condition / loop type:\n    {id, kind, toolkit, toolkit_id, name, description, inputs[], outputs[], notes}\nPure standard library; no platform install needed.\n\"\"\"\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nENTRY = re.compile(r\"^### (.+)$\")\nID_LINE = re.compile(r\"^`([^`]+)` \\((action|condition|loop type)\\)\\s*$\")\nHEAD = re.compile(r\"^# (.+)$\")\nTK_LINE = re.compile(r\"^Toolkit `([^`]+)`, version ([^.]+(?:\\.[^.]+)*)\\.\\s*(.*)$\")\n\n\ndef split_row(line):\n    # table cells; \"\\|\" is an escaped pipe inside a cell\n    cells = re.split(r\"(?<!\\\\)\\|\", line.strip().strip(\"|\"))\n    return [c.strip().replace(\"\\\\|\", \"|\") for c in cells]\n\n\ndef parse_table(lines, i):\n    \"\"\"lines[i] is the header row. Returns (rows as dicts, next index).\"\"\"\n    header = split_row(lines[i])\n    i += 2  # skip the --- separator\n    rows = []\n    while i < len(lines) and lines[i].startswith(\"|\"):\n        cells = split_row(lines[i])\n        rows.append(dict(zip(header, cells)))\n        i += 1\n    return rows, i\n\n\ndef parse_file(path):\n    lines = path.read_text(encoding=\"utf-8\").splitlines()\n    toolkit = \"\"\n    toolkit_id = \"\"\n    entries = []\n    cur = None\n    i = 0\n    while i < len(lines):\n        line = lines[i]\n        m = HEAD.match(line)\n        if m and not toolkit:\n            toolkit = m.group(1)\n        m = TK_LINE.match(line)\n        if m:\n            toolkit_id = m.group(1)\n        m = ENTRY.match(line)\n        if m:\n            cur = {\"name\": m.group(1), \"toolkit\": toolkit, \"toolkit_id\": toolkit_id,\n                   \"inputs\": [], \"outputs\": [], \"description\": \"\", \"notes\": \"\"}\n            entries.append(cur)\n            i += 1\n            continue\n        if cur is not None:\n            m = ID_LINE.match(line)\n            if m:\n                cur[\"id\"], cur[\"kind\"] = m.group(1), m.group(2)\n                # description = next non-empty line\n                j = i + 1\n                while j < len(lines) and not lines[j].strip():\n                    j += 1\n                if j < len(lines) and not lines[j].startswith((\"**\", \"|\", \"#\")):\n                    cur[\"description\"] = lines[j].strip()\n            elif line.strip() == \"**Settings**\":\n                j = i + 1\n                while j < len(lines) and not lines[j].startswith(\"|\"):\n                    j += 1\n                rows, i = parse_table(lines, j)\n                for r in rows:\n                    cur[\"inputs\"].append({\n                        \"name\": r.get(\"Name\", \"\").strip(\"`\"),\n                        \"label\": r.get(\"Field\", \"\"),\n                        \"type\": r.get(\"Type\", \"\"),\n                        \"required\": r.get(\"Required\", \"\") == \"Yes\",\n                        \"default\": r.get(\"Default\", \"\").strip(\"`\"),\n                        \"description\": r.get(\"Description\", \"\"),\n                    })\n                continue\n            elif line.strip() == \"**Results**\":\n                j = i + 1\n                while j < len(lines) and not lines[j].startswith(\"|\"):\n                    j += 1\n                rows, i = parse_table(lines, j)\n                for r in rows:\n                    cur[\"outputs\"].append({\n                        \"name\": r.get(\"Name\", \"\").strip(\"`\"),\n                        \"type\": r.get(\"Type\", \"\"),\n                        \"description\": r.get(\"Description\", \"\"),\n                    })\n                continue\n            elif line.startswith(\"*\") and line.endswith(\"*\") and len(line) > 2:\n                cur[\"notes\"] = line.strip(\"*\")\n        i += 1\n    return [e for e in entries if \"id\" in e]\n\n\ndef main():\n    repo = Path(sys.argv[1])\n    out = Path(sys.argv[2]) if len(sys.argv) > 2 else Path(__file__).with_name(\"catalog.jsonl\")\n    pages = sorted((repo / \"docs/reference/toolkits\").glob(\"*.md\"))\n    pages = [p for p in pages if p.name != \"README.md\"]\n    total = 0\n    with out.open(\"w\", encoding=\"utf-8\") as f:\n        for p in pages:\n            for e in parse_file(p):\n                f.write(json.dumps(e, ensure_ascii=False) + \"\\n\")\n                total += 1\n    print(f\"{total} entries from {len(pages)} toolkit pages -> {out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "build_sft_data.py": "\"\"\"Build seed SFT data for nanochat from the OOA platform repo. No LLM, no network beyond the clone.\n\nUsage: python build_sft_data.py PLATFORM_REPO OUT_DIR [MAX_CHARS]\nWrites OUT_DIR/{catalog.jsonl,train.jsonl,val.jsonl,eval.jsonl}\n  train/val : {\"messages\":[{\"role\":\"user\"|\"assistant\",\"content\":...}]}  (nanochat format)\n  eval      : {\"family\",\"prompt\",\"gold_flow\"}  held-out workflow families, never in train/val\nDialogue types: (1) request -> minified Flow JSON from the gold flows (3 prompt wordings each),\n                (2) catalog lookups: which action does X / what inputs does an action take.\nFlows longer than MAX_CHARS (default 12000) are skipped so they are not truncated at SFT length.\n\"\"\"\nimport glob, hashlib, json, os, random, re, sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\n\ndef family(path):\n    return re.sub(r\"-capture$\", \"\", os.path.basename(path)[:-len(\".flow.json\")])\n\n\ndef is_heldout(fam):\n    return int(hashlib.sha1(fam.encode()).hexdigest(), 16) % 4 == 0\n\n\nWORDINGS = [\n    \"{d}\",\n    \"Create an OOA flow for this: {d}\",\n    \"Build me an automation. {d}\",\n]\n\n\ndef lower_first(s):\n    return s[:1].lower() + s[1:] if s else s\n\n\ndef main():\n    repo, out = Path(sys.argv[1]), Path(sys.argv[2])\n    max_chars = int(sys.argv[3]) if len(sys.argv) > 3 else 12000\n    out.mkdir(parents=True, exist_ok=True)\n\n    # catalog\n    import importlib.util\n    here = Path(__file__).resolve().parent.parent / \"data/catalog/build_catalog.py\"\n    if not here.exists():\n        here = Path(__file__).resolve().parent / \"build_catalog.py\"  # notebook layout\n    spec = importlib.util.spec_from_file_location(\"bc\", here)\n    bc = importlib.util.module_from_spec(spec); spec.loader.exec_module(bc)\n    cat = []\n    for p in sorted((repo / \"docs/reference/toolkits\").glob(\"*.md\")):\n        if p.name != \"README.md\":\n            cat += bc.parse_file(p)\n    with open(out / \"catalog.jsonl\", \"w\", encoding=\"utf-8\") as f:\n        for e in cat:\n            f.write(json.dumps(e, ensure_ascii=False) + \"\\n\")\n\n    train, val, evals = [], [], []\n    rng = random.Random(7)\n    skipped = []\n    for path in sorted(glob.glob(str(repo / \"examples/**/*.flow.json\"), recursive=True)):\n        fam = family(path)\n        d = json.load(open(path, encoding=\"utf-8\"))\n        desc = (d.get(\"description\") or \"\").strip()\n        if not desc or fam == \"make-office-files\":\n            continue\n        flow = json.dumps(d, separators=(\",\", \":\"), ensure_ascii=False)\n        if is_heldout(fam):\n            if not path.endswith(\"-capture.flow.json\"):\n                evals.append({\"family\": fam, \"prompt\": desc, \"gold_flow\": os.path.relpath(path, repo)})\n            continue\n        if len(flow) > max_chars:\n            skipped.append((fam, len(flow)))\n            continue\n        for w in WORDINGS:\n            train.append({\"messages\": [{\"role\": \"user\", \"content\": w.format(d=desc)},\n                                       {\"role\": \"assistant\", \"content\": flow}]})\n\n    # catalog dialogues (actions only)\n    cat_rows = []\n    for e in cat:\n        if e[\"kind\"] != \"action\":\n            continue\n        req = [i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]]\n        opt = [i[\"name\"] for i in e[\"inputs\"] if not i[\"required\"]]\n        desc = e[\"description\"].rstrip(\".\")\n        cat_rows.append({\"messages\": [\n            {\"role\": \"user\", \"content\": f\"Which OOA action would I use to {lower_first(desc)}?\"},\n            {\"role\": \"assistant\", \"content\": f\"Use `{e['id']}` ({e['toolkit']}: {e['name']}).\"}]})\n        cat_rows.append({\"messages\": [\n            {\"role\": \"user\", \"content\": f\"What inputs does {e['id']} take?\"},\n            {\"role\": \"assistant\", \"content\":\n                f\"`{e['id']}` requires: {', '.join(req) if req else 'nothing'}. \"\n                f\"Optional: {', '.join(opt[:12]) if opt else 'none'}.\"}]})\n    rng.shuffle(cat_rows)\n    cut = max(1, len(cat_rows) // 50)\n    val = cat_rows[:cut] + [r for r in train[::17]]\n    train = train + cat_rows[cut:]\n    rng.shuffle(train)\n\n    # separate files so the trainer can weight flows against the (much more numerous) catalog rows\n    flow_rows = [r for r in train if len(r[\"messages\"][1][\"content\"]) > 800 and r[\"messages\"][1][\"content\"].startswith(\"{\")]\n    cat_only = [r for r in train if r not in flow_rows]\n    for name, rows in ((\"train\", train), (\"train_flows\", flow_rows), (\"train_catalog\", cat_only),\n                       (\"val\", val), (\"eval\", evals)):\n        with open(out / f\"{name}.jsonl\", \"w\", encoding=\"utf-8\") as f:\n            for r in rows:\n                f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n    print(f\"catalog {len(cat)} | train {len(train)} val {len(val)} eval {len(evals)} \"\n          f\"| flows skipped for length: {len(skipped)} {skipped[:6]}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "ooa_task.py": "\"\"\"nanochat Task that serves OOA dialogues from a JSONL file.\n\nCopy into nanochat/tasks/ (or add to PYTHONPATH) and use in scripts/chat_sft.py's train_tasks list, e.g.\n    from tasks.ooa import OOADialogues\n    train_tasks = [OOADialogues(\"data/dialogues/train.jsonl\"), ...]\nEach JSONL line: {\"messages\": [{\"role\": \"user\"|\"assistant\", \"content\": str}, ...]}\n - optional first message with role \"system\" (nanochat merges it into the first user turn)\n - roles must alternate user, assistant, ...; assistant content is the minified Flow JSON or a short reply\nFormat checked against nanochat tasks/smoltalk.py and Tokenizer.render_conversation (commit 92d63d4).\n\"\"\"\nimport json\nfrom tasks.common import Task\n\n\nclass OOADialogues(Task):\n    def __init__(self, path, **kwargs):\n        super().__init__(**kwargs)\n        with open(path, encoding=\"utf-8\") as f:\n            self.rows = [json.loads(l) for l in f if l.strip()]\n\n    @property\n    def eval_type(self):\n        return \"generative\"\n\n    def num_examples(self):\n        return len(self.rows)\n\n    def get_example(self, index):\n        messages = self.rows[index][\"messages\"]\n        rest = messages[1:] if messages[0][\"role\"] == \"system\" else messages\n        assert len(rest) >= 2\n        for i, m in enumerate(rest):\n            assert m[\"role\"] == (\"user\" if i % 2 == 0 else \"assistant\")\n            assert isinstance(m[\"content\"], str)\n        return {\"messages\": messages}\n", "gen_flows.py": "\"\"\"Programmatic OOA flow generator: composes valid flows from the action catalog, no LLM.\n\nUsage: python gen_flows.py CATALOG_JSONL PLATFORM_REPO OUT_JSONL [N=3000] [SEED=1]\nEach example: a natural-language request -> minified Flow JSON (nanochat {\"messages\": [...]} format).\nFlows are 1-4 steps. A step's required inputs come from earlier outputs of a matching type when possible,\notherwise they become declared flow inputs. Every flow is checked before it is kept:\n  JSON schema (flow.v1.json), action exists in catalog, input and output names exist, every $variable is declared.\nLimits (v1): chains stay inside one toolkit (so they stay plausible), optionally wrapped in one loop / if / try container, required inputs only, no credential actions.\n\"\"\"\nimport json, random, re, sys\nfrom pathlib import Path\n\nTYPE_MAP = {\"Text\": \"String\", \"File path\": \"String\", \"Number\": \"Number\", \"Yes/No\": \"Boolean\", \"List\": \"List\",\n            \"Table\": \"Table\", \"Dictionary\": \"Dictionary\", \"DateTime\": \"DateTime\", \"Any\": \"Any\"}\nCOMPAT = {\"String\": {\"String\", \"Any\"}, \"Number\": {\"Number\", \"Any\"}, \"Boolean\": {\"Boolean\", \"Any\"},\n          \"List\": {\"List\", \"Any\"}, \"Table\": {\"Table\", \"Any\"}, \"Dictionary\": {\"Dictionary\", \"Any\"},\n          \"DateTime\": {\"DateTime\", \"Any\"}, \"Any\": {\"Any\", \"String\", \"Number\", \"Boolean\", \"List\", \"Table\", \"Dictionary\", \"DateTime\"}}\nWORDINGS = [\"{c}\", \"Create an OOA flow: {c}\", \"Build an automation that does this. {c}\"]\n\n\ndef vtype(t):\n    return TYPE_MAP.get(t)\n\n\ndef snake(s):\n    s = re.sub(r\"[^A-Za-z0-9]+\", \"_\", s).strip(\"_\").lower()\n    return s if s and not s[0].isdigit() else \"v_\" + s\n\n\ndef fillable(e):\n    if e[\"toolkit_id\"] == \"core\":   # control-flow built-ins (continue, break, ...) only make sense inside containers\n        return False\n    req = [i for i in e[\"inputs\"] if i[\"required\"]]\n    return (e[\"kind\"] == \"action\" and len(req) <= 6 and all(vtype(i[\"type\"]) for i in req)\n            and all(vtype(o[\"type\"]) or True for o in e[\"outputs\"]))\n\n\ndef clean_desc(d):\n    return d.strip().rstrip(\".\").strip()\n\n\n# request wordings: full catalog description, its first clause, the short action name, or a numbered list\nSTYLES = {\"full\": 40, \"brief\": 25, \"short\": 20, \"numbered\": 15}\n\n\ndef first_clause(d):\n    return re.split(r\"; |\\. | \\(|, optionally |, or |: \", d, maxsplit=1)[0].strip()\n\n\ndef phrase(e, desc, style):\n    if style == \"full\":\n        return desc\n    if style == \"short\":\n        return f\"{e['name'][:1].lower() + e['name'][1:]} ({e['toolkit']})\"\n    return first_clause(desc)\n\n\ndef build(cat_actions, rng, cfg=None):\n    cfg = cfg or {}\n    n_steps = rng.choices([1, 2, 3, 4], weights=cfg.get(\"step_weights\", [25, 35, 25, 15]))[0]\n    pat = cfg.get(\"patterns\")          # learned from real flows: which action follows which (see real_patterns)\n    if pat:\n        n_steps = rng.choice(pat[\"lengths\"])\n    variables, steps, used_names = [], [], set()\n    avail = []  # (var_name, vtype) outputs not yet consumed\n    descs, in_names = [], []\n\n    def new_name(base):\n        base = snake(base)[:40]\n        n, k = base, 2\n        while n in used_names:\n            n, k = f\"{base}_{k}\", k + 1\n        used_names.add(n)\n        return n\n\n    prev = None\n    descs_steps, used = [], []\n    for si in range(n_steps):\n        e = None\n        if pat:\n            pool = pat[\"starters\"] if prev is None else pat[\"next\"].get(prev[\"id\"], [])\n            pool = [pat[\"by_id\"][a] for a in pool if a in pat[\"by_id\"]]\n            if pool:\n                e = rng.choice(pool)        # repeats in the list = how often real flows make that move\n        cands = cat_actions\n        if prev is not None and avail:\n            want = {t for _, t in avail}\n            c2 = [e for e in cat_actions if any(vtype(i[\"type\"]) in want for i in e[\"inputs\"] if i[\"required\"])]\n            cands = c2 or cat_actions\n        if prev is not None:   # chains stay inside one toolkit so they stay plausible\n            cands = [c for c in cat_actions if c[\"toolkit_id\"] == prev[\"toolkit_id\"]\n                     and (not avail or any(vtype(i[\"type\"]) in {t for _, t in avail}\n                                           for i in c[\"inputs\"] if i[\"required\"]))]\n            if not cands and e is None:\n                break\n        if e is None:\n            e = rng.choice(cands)\n        args = {}\n        for i in [i for i in e[\"inputs\"] if i[\"required\"]]:\n            t = vtype(i[\"type\"])\n            src = None if t == \"Any\" else next((a for a in avail if a[1] in COMPAT[t] and a[1] != \"Any\"), None)\n            if src:\n                args[i[\"name\"]] = \"$\" + src[0]\n                avail.remove(src)\n            else:\n                vn = new_name(i[\"name\"])\n                var = {\"name\": vn, \"type\": t, \"direction\": \"input\", \"required\": True}\n                if i[\"description\"]:\n                    var[\"description\"] = i[\"description\"][:120]\n                variables.append(var)\n                in_names.append(vn)\n                args[i[\"name\"]] = \"$\" + vn\n        out = {}\n        for o in e[\"outputs\"][:2]:\n            vn = new_name(o[\"name\"])\n            out[o[\"name\"]] = vn\n            variables.append({\"name\": vn, \"type\": vtype(o[\"type\"]) or \"Any\", \"direction\": \"local\"})\n            avail.append((vn, vtype(o[\"type\"]) or \"Any\"))\n        step = {\"id\": f\"s{si + 1}\", \"action\": e[\"id\"], \"description\": clean_desc(e[\"description\"])[:140], \"args\": args}\n        if out:\n            step[\"out\"] = out\n        steps.append(step)\n        descs_steps.append(step)\n        descs.append(clean_desc(e[\"description\"]))\n        used.append(e)\n        prev = e\n    # optionally wrap the chain in a control-flow container (loop / if / try)\n    wrap_kind = rng.choices([\"none\", \"loop\", \"if\", \"try\"], weights=cfg.get(\"wrap_weights\", [55, 17, 14, 14]))[0]\n    prefix, suffix = \"\", \"\"\n    body = steps\n    if wrap_kind == \"loop\":\n        items = new_name(\"items\"); item = new_name(\"item\")\n        variables.append({\"name\": items, \"type\": \"List\", \"direction\": \"input\", \"required\": True,\n                          \"description\": \"The items to process, one per loop pass.\"})\n        variables.append({\"name\": item, \"type\": \"Any\", \"direction\": \"local\"})\n        in_names.append(items)\n        body = [{\"id\": \"w1\", \"action\": \"core/loop\", \"description\": \"Repeat for each item\",\n                 \"args\": {\"iterator\": \"list\", \"over\": \"$\" + items, \"as\": item}, \"children\": steps}]\n        prefix = f\"For each entry in {items}: \"\n    elif wrap_kind == \"if\":\n        flag = new_name(rng.choice([\"proceed\", \"enabled\", \"send_it\", \"is_active\"]))\n        variables.append({\"name\": flag, \"type\": \"Boolean\", \"direction\": \"input\", \"required\": True,\n                          \"description\": \"Run the steps only when true.\"})\n        in_names.append(flag)\n        body = [{\"id\": \"w1\", \"action\": \"core/if\", \"description\": \"Only when the flag is true\",\n                 \"args\": {\"condition\": \"$\" + flag}, \"children\": steps}]\n        prefix = f\"Only when {flag} is true: \"\n    elif wrap_kind == \"try\":\n        body = [{\"id\": \"w1\", \"action\": \"core/try\", \"description\": \"Handle errors\", \"children\": steps,\n                 \"catch\": [{\"codes\": [], \"children\": [{\"id\": \"w2\", \"action\": \"core/log\", \"args\": {\n                     \"message\": rng.choice([\"Step failed\", \"Something went wrong\", \"Error in this flow\"]),\n                     \"level\": \"error\"}}]}]}]\n        suffix = \" If anything fails, write an error to the log.\"\n    # sequential, unique ids over the whole tree\n    counter = [0]\n    def renumber(nodes):\n        for n in nodes:\n            counter[0] += 1\n            n[\"id\"] = f\"s{counter[0]}\"\n            for k in (\"children\", \"else\", \"finally\"):\n                if isinstance(n.get(k), list):\n                    renumber(n[k])\n            for c in n.get(\"catch\") or []:\n                renumber(c.get(\"children\", []))\n    renumber(body)\n    steps = body\n    # last step's outputs become flow outputs\n    out_names = []\n    last_chain = descs_steps[-1] if descs_steps else None\n    if last_chain and \"out\" in last_chain:\n        for vn in last_chain[\"out\"].values():\n            for v in variables:\n                if v[\"name\"] == vn:\n                    v[\"direction\"] = \"output\"\n                    out_names.append(vn)\n    # how the request is worded: real users do not paste catalog descriptions, so vary the wording\n    style = rng.choices(list(STYLES), weights=[cfg.get(\"prompt_styles\", STYLES)[k] for k in STYLES])[0]\n    parts = [phrase(e, d, style) for e, d in zip(used, descs)]\n    if style == \"numbered\":\n        chain = \"\\n\".join(f\"{k + 1}. {p[:1].upper() + p[1:]}\" for k, p in enumerate(parts))\n        if prefix:\n            chain = prefix.rstrip() + \"\\n\" + chain\n    else:\n        joiner = \", then \" if style == \"full\" else rng.choice([\", then \", \"; \", \", and then \", \". Then \"])\n        chain = joiner.join(p[:1].lower() + p[1:] if k else p[:1].upper() + p[1:] for k, p in enumerate(parts)) + \".\"\n        if prefix:\n            chain = prefix + chain[:1].lower() + chain[1:]\n    chain += suffix\n    extra = \"\"\n    io_prob = cfg.get(\"io_prob\", 1.0 if style == \"full\" else 0.5)    # the \"takes / returns\" sentence is often left out\n    if in_names and rng.random() < io_prob:\n        extra += f\" The flow takes {', '.join(in_names)} as input.\"\n    if out_names and rng.random() < io_prob:\n        extra += f\" It returns {', '.join(out_names)}.\"\n    first = descs[0]\n    name = \" \".join(first.split()[:6]).title()\n    flow = {\"kind\": \"TaskBot\", \"id\": snake(first)[:40] or \"flow\", \"name\": name,\n            \"description\": chain, \"variables\": variables, \"steps\": steps}\n    return flow, chain + extra\n\n\ndef real_patterns(repo, catalog, max_len=10):\n    \"\"\"Action-to-action moves seen in the REAL example flows (excluding held-out test families), as a generator config.\n    Real flows are long and cross toolkits (browser -> excel -> email); this lets generated flows do the same.\"\"\"\n    import glob, hashlib, os\n    by_id = {e[\"id\"]: e for e in catalog if fillable(e)}\n    nxt, starters, lengths = {}, [], []\n    for path in sorted(glob.glob(str(Path(repo) / \"examples\" / \"**\" / \"*.flow.json\"), recursive=True)):\n        fam = re.sub(r\"-capture$\", \"\", os.path.basename(path)[:-len(\".flow.json\")])\n        if int(hashlib.sha1(fam.encode()).hexdigest(), 16) % 4 == 0:      # same rule as build_sft_data: held out\n            continue\n        flow = json.load(open(path, encoding=\"utf-8\"))\n        seq = []\n        def walk(ns):\n            for n in ns or []:\n                if isinstance(n, dict):\n                    if not str(n.get(\"action\", \"\")).startswith(\"core/\"):\n                        seq.append(n.get(\"action\"))\n                    for k in (\"children\", \"else\", \"finally\"):\n                        if isinstance(n.get(k), list):\n                            walk(n[k])\n                    for c in n.get(\"catch\") or []:\n                        walk(c.get(\"children\"))\n        walk(flow.get(\"steps\"))\n        if len(seq) < 2:\n            continue\n        starters.append(seq[0])\n        lengths.append(max(3, min(max_len, len(seq))))\n        for a, b in zip(seq, seq[1:]):\n            nxt.setdefault(a, []).append(b)\n    return {\"next\": nxt, \"starters\": starters, \"lengths\": lengths, \"by_id\": by_id}\n\n\ndef make_validator(repo, catalog):\n    import jsonschema\n    schema = json.load(open(Path(repo) / \"contracts/flow-schema/flow.v1.json\"))\n    cat = {e[\"id\"]: e for e in catalog}\n    jsonschema.Draft202012Validator.check_schema(schema)\n    schema_validator = jsonschema.validators.validator_for(schema)(schema)   # built once; validate() rebuilt it for every flow\n\n    def validate(flow, check_schema=True):\n        if check_schema:\n            schema_validator.validate(flow)\n        declared = {v[\"name\"] for v in flow[\"variables\"]}\n        def walk(nodes):\n            for n in nodes:\n                yield n\n                for k in (\"children\", \"else\", \"finally\"):\n                    if isinstance(n.get(k), list):\n                        yield from walk(n[k])\n                for c in n.get(\"catch\") or []:\n                    yield from walk(c.get(\"children\", []))\n        ids = []\n        for s in walk(flow[\"steps\"]):\n            ids.append(s[\"id\"])\n            if s[\"action\"].startswith(\"core/\"):\n                for v in s.get(\"args\", {}).values():\n                    if isinstance(v, str) and v.startswith(\"$\"):\n                        assert v[1:] in declared, f\"undeclared {v}\"\n                continue\n            e = cat[s[\"action\"]]\n            ins = {i[\"name\"] for i in e[\"inputs\"]}\n            outs = {o[\"name\"] for o in e[\"outputs\"]}\n            assert set(s.get(\"args\", {})) <= ins, f\"unknown input in {s['action']}\"\n            assert set(s.get(\"out\", {})) <= outs, f\"unknown output in {s['action']}\"\n            req = {i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]}\n            assert req <= set(s.get(\"args\", {})), \"missing required input\"\n            for v in s.get(\"args\", {}).values():\n                if isinstance(v, str) and v.startswith(\"$\"):\n                    assert v[1:] in declared, f\"undeclared {v}\"\n            for v in s.get(\"out\", {}).values():\n                assert v in declared, f\"undeclared out {v}\"\n        assert len(ids) == len(set(ids)), \"duplicate step ids\"\n    return validate\n\n\ndef generate(catalog, repo, n, seed, exclude=(), schema_every=1, cfg=None, max_chars=None):\n    \"\"\"Return n unique valid (request, flow_dict, minified_flow) triples; skips minified flows in `exclude`.\n    Every flow is fully checked (schema, actions, inputs, outputs, variables, ids). schema_every=N checks the schema on\n    every N-th candidate only. (The slow part used to be jsonschema.validate rebuilding its validator on every call.)\"\"\"\n    rng = random.Random(seed)\n    actions = [e for e in catalog if fillable(e)]\n    validate = make_validator(repo, catalog)\n    seen, out, tries = set(exclude), [], 0\n    while len(out) < n and tries < n * 30:\n        tries += 1\n        flow, req = build(actions, rng, cfg)\n        try:\n            validate(flow, check_schema=(tries % schema_every == 0))\n        except Exception:\n            continue\n        mini = json.dumps(flow, separators=(\",\", \":\"), ensure_ascii=False)\n        if mini in seen:\n            continue\n        q = rng.choice(WORDINGS).format(c=req)\n        if max_chars and len(q) + len(mini) > max_chars:   # rows longer than the training window never fit nanochat's packing\n            continue\n        seen.add(mini)\n        out.append((q, flow, mini))\n    return out\n\n\ndef main():\n    if len(sys.argv) > 6 and sys.argv[6] == \"patterns\":     # real-flow patterns: long, multi-toolkit flows\n        cat_path, repo, out, n, seed = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4]), int(sys.argv[5])\n        catalog = [json.loads(l) for l in open(cat_path, encoding=\"utf-8\")]\n        rows = generate(catalog, repo, n, seed, cfg={\"patterns\": real_patterns(repo, catalog)}, max_chars=5500)\n        with open(out, \"w\", encoding=\"utf-8\") as f:\n            for q, _, mini in rows:\n                f.write(json.dumps({\"messages\": [{\"role\": \"user\", \"content\": q}, {\"role\": \"assistant\", \"content\": mini}]}, ensure_ascii=False) + \"\\n\")\n        lens = sorted(len(m) for *_, m in rows)\n        print(f\"{len(rows)} real-pattern flows | chars median {lens[len(lens)//2]} max {lens[-1]}\")\n        return\n    cat_path, repo, out = sys.argv[1], sys.argv[2], sys.argv[3]\n    n = int(sys.argv[4]) if len(sys.argv) > 4 else 3000\n    rng = random.Random(int(sys.argv[5]) if len(sys.argv) > 5 else 1)\n    catalog = [json.loads(l) for l in open(cat_path, encoding=\"utf-8\")]\n    actions = [e for e in catalog if fillable(e)]\n    validate = make_validator(repo, catalog)\n    seen, rows, rejected, lens = set(), [], 0, []\n    tries = 0\n    while len(rows) < n and tries < n * 20:\n        tries += 1\n        flow, req = build(actions, rng)\n        try:\n            validate(flow)\n        except Exception:\n            rejected += 1\n            continue\n        mini = json.dumps(flow, separators=(\",\", \":\"), ensure_ascii=False)\n        if mini in seen:\n            continue\n        seen.add(mini)\n        lens.append(len(mini))\n        rows.append({\"messages\": [{\"role\": \"user\", \"content\": rng.choice(WORDINGS).format(c=req)},\n                                  {\"role\": \"assistant\", \"content\": mini}]})\n    with open(out, \"w\", encoding=\"utf-8\") as f:\n        for r in rows:\n            f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n    lens.sort()\n    print(f\"{len(rows)} flows from {len(actions)} fillable actions | rejected {rejected} | chars median {lens[len(lens)//2]} max {lens[-1]}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "ooa_reward.py": "\"\"\"Verifiable reward for OOA flows: the platform's own rules score the model's answer. No human, no LLM judge.\n\nscore(text, gold_actions=None) -> {\"total\": 0..1, \"solved\": bool, parts...}\nParts (weights): starts_obj .05 | json .20 | schema .15 | actions_exist .15 | args_ok .10 | vars_ok .10 | match .25\n - match compares the action sequence with the gold flow (partial credit by sequence similarity, so there is\n   a gradient to climb even when the answer is not exact). Without gold, total is capped at .75.\n - A flow with no steps scores 0 on everything after json: empty flows cannot win.\n - solved = every part passes and (if gold given) the action sequence matches exactly.\nNeeds env OOA_CATALOG (catalog.jsonl) and OOA_SCHEMA (flow.v1.json), or call configure().\n\"\"\"\nimport difflib, json, os, re\n\n_CAT = None\n_SCHEMA = None\n_SV = None   # schema validator, built once (jsonschema.validate() rebuilds it on every call: about 70 ms each)\n\n\ndef configure(catalog_path=None, schema_path=None):\n    global _CAT, _SCHEMA, _SV\n    catalog_path = catalog_path or os.environ[\"OOA_CATALOG\"]\n    schema_path = schema_path or os.environ[\"OOA_SCHEMA\"]\n    _CAT = {}\n    for l in open(catalog_path, encoding=\"utf-8\"):\n        e = json.loads(l)\n        _CAT[e[\"id\"]] = e\n    _SCHEMA = json.load(open(schema_path, encoding=\"utf-8\"))\n    import jsonschema\n    _SV = jsonschema.validators.validator_for(_SCHEMA)(_SCHEMA)\n\n\ndef walk(nodes):\n    for n in nodes or []:\n        if not isinstance(n, dict):\n            continue\n        yield n\n        for k in (\"children\", \"else\", \"finally\"):\n            if isinstance(n.get(k), list):\n                yield from walk(n[k])\n        for ei in n.get(\"elseIf\") or []:\n            if isinstance(ei, dict):\n                yield from walk(ei.get(\"children\"))\n        for c in n.get(\"catch\") or []:\n            if isinstance(c, dict):\n                yield from walk(c.get(\"children\"))\n\n\ndef actions_of(flow):\n    return [s.get(\"action\") for s in walk(flow.get(\"steps\"))]\n\n\ndef gold_actions(flow_text):\n    return actions_of(json.loads(flow_text))\n\n\n# variable references in arg values (platform docs/llms.txt): \"$a.b[0]\" reference, \"text ${name} ${len($xs)}\"\n# interpolation, {\"expr\": \"$a * 2\"} expression, {\"literal\": \"$x\"} escape (not a reference). sys / global are built in.\n_REF_PLAIN = re.compile(r\"\\$(?!\\{)([A-Za-z_]\\w*)\")\n_REF_TEMPLATE = re.compile(r\"\\$\\{\\s*([A-Za-z_]\\w*)(?![\\w(])\")\nBUILTIN_ROOTS = {\"sys\", \"global\"}\n\n\ndef refs(value):\n    \"\"\"Root variable names an arg value reads, e.g. \"${folder}/data.csv\" -> {\"folder\"}, \"$row.Name\" -> {\"row\"}.\"\"\"\n    out = set()\n    if isinstance(value, str):\n        out.update(_REF_PLAIN.findall(value))\n        out.update(_REF_TEMPLATE.findall(value))\n    elif isinstance(value, dict):\n        if set(value) == {\"literal\"}:\n            return out\n        for v in value.values():\n            out |= refs(v)\n    elif isinstance(value, list):\n        for v in value:\n            out |= refs(v)\n    return out - BUILTIN_ROOTS\n\n\ndef clean(text):\n    return text.split(\"<|assistant_end|>\")[0].strip()\n\n\nW = {\"starts_obj\": .05, \"json\": .20, \"schema\": .15, \"actions_exist\": .15, \"args_ok\": .10, \"vars_ok\": .10, \"match\": .25}\n\n\ndef score(text, gold=None):\n    if _CAT is None:\n        configure()\n    import jsonschema\n    text = clean(text)\n    p = {k: 0.0 for k in W}\n    p[\"starts_obj\"] = 1.0 if text.startswith(\"{\") else 0.0\n    try:\n        flow = json.loads(text)\n        assert isinstance(flow, dict)\n        p[\"json\"] = 1.0\n    except Exception:\n        return _total(p, gold)\n    steps = list(walk(flow.get(\"steps\")))\n    if not steps:\n        return _total(p, gold)\n    try:\n        _SV.validate(flow)\n        p[\"schema\"] = 1.0\n    except Exception:\n        pass\n    acts = [s.get(\"action\") for s in steps]\n    p[\"actions_exist\"] = 1.0 if all(a in _CAT for a in acts) else 0.0\n    ok = p[\"actions_exist\"] == 1.0\n    declared = {v.get(\"name\") for v in flow.get(\"variables\", []) if isinstance(v, dict)}\n    ids = [s.get(\"id\") for s in steps]\n    vars_ok = len(ids) == len(set(ids))\n    for s in steps:\n        args, out = s.get(\"args\") or {}, s.get(\"out\") or {}\n        if not isinstance(args, dict) or not isinstance(out, dict):\n            ok = vars_ok = False\n            continue\n        e = _CAT.get(s.get(\"action\"))\n        if e and not s[\"action\"].startswith(\"core/\"):\n            ins = {i[\"name\"] for i in e[\"inputs\"]}\n            outs = {o[\"name\"] for o in e[\"outputs\"]}\n            req = {i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]}\n            if not (set(args) <= ins and set(out) <= outs and req <= set(args)):\n                ok = False\n        if refs(args) - declared:\n            vars_ok = False\n        for v in out.values():\n            if v not in declared:\n                vars_ok = False\n    p[\"args_ok\"] = 1.0 if ok else 0.0\n    p[\"vars_ok\"] = 1.0 if vars_ok else 0.0\n    if gold is not None:\n        p[\"match\"] = difflib.SequenceMatcher(a=acts, b=gold).ratio()\n    return _total(p, gold)\n\n\ndef _total(p, gold):\n    total = sum(W[k] * p[k] for k in W)\n    valid = all(p[k] == 1.0 for k in (\"json\", \"schema\", \"actions_exist\", \"args_ok\", \"vars_ok\"))\n    solved = valid and (p[\"match\"] == 1.0 if gold is not None else True)\n    return {\"total\": round(total, 4), \"solved\": solved, \"valid\": valid, **p}\n\n\ndef problems(text, limit=8):\n    \"\"\"Plain-language list of what is wrong with an answer (empty list = valid). Used to tell a model how to fix a retry.\"\"\"\n    if _CAT is None:\n        configure()\n    text = clean(text)\n    try:\n        flow = json.loads(text)\n        assert isinstance(flow, dict)\n    except Exception:\n        return [\"The answer is not a single valid JSON object.\"]\n    out = []\n    for e in list(_SV.iter_errors(flow))[:3]:\n        out.append(\"Schema: \" + e.message[:140] + (\" at \" + \"/\".join(map(str, e.path)) if e.path else \"\"))\n    steps = list(walk(flow.get(\"steps\")))\n    if not steps:\n        out.append(\"The flow has no steps.\")\n    declared = {v.get(\"name\") for v in flow.get(\"variables\", []) if isinstance(v, dict)}\n    ids = [s.get(\"id\") for s in steps]\n    if len(ids) != len(set(ids)):\n        out.append(\"Step ids are not unique.\")\n    for s in steps:\n        a = s.get(\"action\")\n        e = _CAT.get(a)\n        if e is None:\n            out.append(f\"Unknown action {a!r}.\")\n            continue\n        args, o = s.get(\"args\") or {}, s.get(\"out\") or {}\n        if not a.startswith(\"core/\"):\n            ins = {i[\"name\"] for i in e[\"inputs\"]}\n            outs = {x[\"name\"] for x in e[\"outputs\"]}\n            req = {i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]}\n            for k in args:\n                if k not in ins:\n                    out.append(f\"{a} has no input {k!r}. Inputs: {', '.join(sorted(ins)) or '-'}.\")\n            for k in o:\n                if k not in outs:\n                    out.append(f\"{a} has no output {k!r}. Outputs: {', '.join(sorted(outs)) or '-'}.\")\n            for k in sorted(req - set(args)):\n                out.append(f\"{a} is missing required input {k!r}.\")\n        for v in sorted(refs(args) - declared):\n            out.append(f\"Variable {v!r} is used but not declared in variables.\")\n        for v in o.values():\n            if v not in declared:\n                out.append(f\"Output variable {v!r} is not declared in variables.\")\n    return list(dict.fromkeys(out))[:limit]\n", "ooa_rl_task.py": "\"\"\"nanochat Task for reinforcement learning on OOA flows (copy to nanochat/tasks/ooa_rl.py).\nRows: JSONL {\"messages\":[{\"role\":\"user\",...},{\"role\":\"assistant\",\"content\":<gold flow json>}]}.\nnanochat's chat_rl.py samples several answers per request and calls reward(); answers the verifier scores higher than\nthe group average are reinforced (REINFORCE with advantage = reward - mean). evaluate() = fully solved (pass@k).\n\"\"\"\nimport json, sys\nfrom tasks.common import Task\n\n\nclass OOAFlowRL(Task):\n    def __init__(self, path, ooa_dir=\"/kaggle/working/ooa\", **kwargs):\n        super().__init__(**kwargs)\n        if ooa_dir not in sys.path:\n            sys.path.insert(0, ooa_dir)\n        import ooa_reward\n        self.R = ooa_reward\n        self.rows = [json.loads(l) for l in open(path, encoding=\"utf-8\") if l.strip()]\n\n    @property\n    def eval_type(self):\n        return \"generative\"\n\n    def num_examples(self):\n        return len(self.rows)\n\n    def get_example(self, index):\n        return {\"messages\": self.rows[index][\"messages\"]}\n\n    def _score(self, conversation, response):\n        gold = self.R.gold_actions(conversation[\"messages\"][-1][\"content\"])\n        return self.R.score(response, gold)\n\n    def evaluate(self, conversation, assistant_response):\n        return int(self._score(conversation, assistant_response)[\"solved\"])\n\n    def reward(self, conversation, assistant_response):\n        return float(self._score(conversation, assistant_response)[\"total\"])\n", "ooa_self_improve.py": "\"\"\"Self-improvement round for the OOA model (expert iteration with a verifier). Copy to nanochat/scripts/.\n\nEach round:\n  1. make FRESH requests (new generator seed) plus replay the requests it failed on before\n  2. the current model writes K candidate flows per request\n  3. the verifier (ooa_reward) scores every candidate against the platform rules and the known-good action sequence\n  4. solved -> keep the model's own best answer in self_improved.jsonl (more data in its own successful style)\n     never solved -> keep the verified gold flow in hard_fixed.jsonl (learn from mistakes) and queue it for replay\n  5. log pass rate, so progress is measured on fresh requests every round\nThe next SFT run trains from the pretrained base on base data + both files (never fine-tune only on its own output:\nthat drifts and collapses). Usage:\n  python -m scripts.ooa_self_improve --source sft --out DIR --seed 11 --num-prompts 200 --samples 8\n\"\"\"\nimport argparse, json, os, sys, time\nimport torch\nfrom nanochat.common import compute_init, autodetect_device_type\nfrom nanochat.checkpoint_manager import load_model\nfrom nanochat.engine import Engine\n\nap = argparse.ArgumentParser()\nap.add_argument(\"--source\", default=\"sft\")\nap.add_argument(\"--out\", required=True, help=\"dir holding catalog.jsonl; self-improvement files are written here\")\nap.add_argument(\"--platform\", default=\"/kaggle/working/platform\")\nap.add_argument(\"--ooa-dir\", default=\"/kaggle/working/ooa\")\nap.add_argument(\"--seed\", type=int, default=11)\nap.add_argument(\"--num-prompts\", type=int, default=200)\nap.add_argument(\"--samples\", type=int, default=8)\nap.add_argument(\"--max-new-tokens\", type=int, default=1024)\nap.add_argument(\"--temperature\", type=float, default=0.8)\nap.add_argument(\"--top-k\", type=int, default=50)\nap.add_argument(\"--replay\", type=int, default=50, help=\"max earlier failures to retry this round\")\nargs = ap.parse_args()\n\nsys.path.insert(0, args.ooa_dir)\nos.environ[\"OOA_CATALOG\"] = os.path.join(args.out, \"catalog.jsonl\")\nos.environ[\"OOA_SCHEMA\"] = os.path.join(args.platform, \"contracts/flow-schema/flow.v1.json\")\nimport ooa_reward as R\nimport gen_flows as G\n\ncatalog = [json.loads(l) for l in open(os.environ[\"OOA_CATALOG\"], encoding=\"utf-8\")]\n\n\ndef read(path):\n    return [json.loads(l) for l in open(path, encoding=\"utf-8\")] if os.path.exists(path) else []\n\n\n# never reuse a flow the model already trained on or was tested on\nseen = set()\nfor f in (\"synth_train.jsonl\", \"self_improved.jsonl\", \"hard_fixed.jsonl\", \"eval_synth.jsonl\", \"rl_val.jsonl\"):\n    for r in read(os.path.join(args.out, f)):\n        if \"messages\" in r:\n            seen.add(r[\"messages\"][1][\"content\"])\nfresh = G.generate(catalog, args.platform, args.num_prompts, args.seed, exclude=seen)\ntasks = [{\"prompt\": q, \"gold\": mini} for q, _, mini in fresh]\nhard_path = os.path.join(args.out, \"hard_queue.jsonl\")\nqueue = read(hard_path)[-args.replay:]\ntasks += [{\"prompt\": h[\"prompt\"], \"gold\": h[\"gold\"]} for h in queue]\nprint(f\"{len(fresh)} fresh requests + {len(queue)} replayed failures\")\n\ndevice_type = autodetect_device_type()\n_, _, _, _, device = compute_init(device_type)\nmodel, tokenizer, meta = load_model(args.source, device, phase=\"eval\")\nengine = Engine(model, tokenizer)\nmodel.eval()\n\nnew_good, new_hard, solved_fresh, solved_all, still_hard = [], [], 0, 0, []\nrewards = []\nparts = {k: 0.0 for k in (\"starts_obj\", \"json\", \"schema\", \"actions_exist\", \"args_ok\", \"vars_ok\", \"match\")}\nn_samples = 0\nt0 = time.time()\nfor ti, t in enumerate(tasks):\n    conv = {\"messages\": [{\"role\": \"user\", \"content\": t[\"prompt\"]}, {\"role\": \"assistant\", \"content\": t[\"gold\"]}]}\n    tokens = tokenizer.render_for_completion(conv)\n    gold = R.gold_actions(t[\"gold\"])\n    seqs, _ = engine.generate_batch(tokens, num_samples=args.samples, max_tokens=args.max_new_tokens,\n                                    temperature=args.temperature, top_k=args.top_k, seed=args.seed * 1000 + ti)\n    scored = []\n    for s in seqs:\n        text = R.clean(tokenizer.decode(s[len(tokens):]))\n        scored.append((R.score(text, gold), text))\n    for sc, _ in scored:\n        n_samples += 1\n        for k in parts:\n            parts[k] += sc[k]\n    best = max(scored, key=lambda x: x[0][\"total\"])\n    rewards.append(sum(x[0][\"total\"] for x in scored) / len(scored))\n    is_fresh = ti < len(fresh)\n    if best[0][\"solved\"]:\n        solved_all += 1\n        solved_fresh += is_fresh\n        new_good.append({\"messages\": [{\"role\": \"user\", \"content\": t[\"prompt\"]},\n                                      {\"role\": \"assistant\", \"content\": best[1]}]})\n    else:\n        new_hard.append({\"messages\": conv[\"messages\"]})\n        still_hard.append({\"prompt\": t[\"prompt\"], \"gold\": t[\"gold\"]})\n    if (ti + 1) % 25 == 0:\n        print(f\"  {ti + 1}/{len(tasks)} | solved so far {solved_all} | {time.time() - t0:.0f}s\")\n\n\ndef append_unique(name, rows):\n    path = os.path.join(args.out, name)\n    have = {r[\"messages\"][1][\"content\"] for r in read(path)}\n    add = [r for r in rows if r[\"messages\"][1][\"content\"] not in have]\n    with open(path, \"a\", encoding=\"utf-8\") as f:\n        for r in add:\n            f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n    return len(add)\n\na = append_unique(\"self_improved.jsonl\", new_good)\nb = append_unique(\"hard_fixed.jsonl\", new_hard)\nwith open(hard_path, \"w\", encoding=\"utf-8\") as f:   # failures still unsolved are retried next round\n    for h in still_hard:\n        f.write(json.dumps(h, ensure_ascii=False) + \"\\n\")\nstats = {\"time\": time.strftime(\"%Y-%m-%d %H:%M:%S\"), \"seed\": args.seed, \"source\": args.source,\n         \"fresh_requests\": len(fresh), \"fresh_pass_at_k\": round(solved_fresh / max(1, len(fresh)), 3),\n         \"samples\": args.samples, \"mean_reward\": round(sum(rewards) / max(1, len(rewards)), 3),\n         \"parts\": {k: round(v / max(1, n_samples), 3) for k, v in parts.items()},\n         \"added_self_improved\": a, \"added_hard_fixed\": b, \"hard_queue\": len(still_hard)}\nwith open(os.path.join(args.out, \"selfimprove_log.jsonl\"), \"a\") as f:\n    f.write(json.dumps(stats) + \"\\n\")\nprint(\"ROUND RESULT:\", json.dumps(stats))\n", "ingest_feedback.py": "\"\"\"Turn real usage into training data (the long-run self-learning signal).\n\nStudio (or any client) appends one JSON line per suggestion to feedback.jsonl:\n  {\"prompt\": \"...\", \"flow\": {...flow json...}, \"accepted\": true|false, \"edited_flow\": {...optional...}}\nRules (humans are a signal, the verifier is the gate):\n  accepted, or edited_flow given -> candidate training example (edited_flow wins: it is the human's correction)\n  every candidate must pass the verifier (schema, actions exist, args, variables) or it is dropped\n  rejected with no edit -> not trained on; listed in rejected_prompts.jsonl as hard cases to write good answers for\nUsage: python ingest_feedback.py FEEDBACK.jsonl OUT_DIR CATALOG.jsonl FLOW_SCHEMA.json\nWrites OUT_DIR/feedback_train.jsonl (nanochat format) and OUT_DIR/rejected_prompts.jsonl\n\"\"\"\nimport json, os, sys\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfb, out, cat, schema = sys.argv[1:5]\nos.environ[\"OOA_CATALOG\"], os.environ[\"OOA_SCHEMA\"] = cat, schema\nimport ooa_reward as R\n\ngood, rejected, dropped = [], [], 0\nfor l in open(fb, encoding=\"utf-8\"):\n    if not l.strip():\n        continue\n    r = json.loads(l)\n    flow = r.get(\"edited_flow\") or (r.get(\"flow\") if r.get(\"accepted\") else None)\n    if flow is None:\n        rejected.append({\"prompt\": r[\"prompt\"]})\n        continue\n    mini = json.dumps(flow, separators=(\",\", \":\"), ensure_ascii=False)\n    if not R.score(mini)[\"valid\"]:\n        dropped += 1\n        continue\n    good.append({\"messages\": [{\"role\": \"user\", \"content\": r[\"prompt\"]}, {\"role\": \"assistant\", \"content\": mini}]})\nos.makedirs(out, exist_ok=True)\nwith open(os.path.join(out, \"feedback_train.jsonl\"), \"w\", encoding=\"utf-8\") as f:\n    for g in good:\n        f.write(json.dumps(g, ensure_ascii=False) + \"\\n\")\nwith open(os.path.join(out, \"rejected_prompts.jsonl\"), \"w\", encoding=\"utf-8\") as f:\n    for g in rejected:\n        f.write(json.dumps(g, ensure_ascii=False) + \"\\n\")\nprint(f\"{len(good)} usable, {dropped} dropped (failed verifier), {len(rejected)} rejected-without-edit\")\n", "autoresearch/constants.py": "\"\"\"FROZEN settings shared by every experiment. Changing these makes old results incomparable. Agents must not edit this file.\"\"\"\nFIXED_TOKEN_BUDGET = 12_000_000     # nominal budget. NOTE: nanochat counts iterations in micro-batches, so what is really trained is\n                                    # budget / grad_accum = 3.0M tokens = 92 optimizer steps. Kept as is so earlier results stay comparable.\nKEEP_EPSILON = 0.003                # an experiment must beat the best score by more than this to be kept (noise guard)\nEVAL_SAMPLES = 4                    # answers sampled per frozen-eval request\nEVAL_TEMPERATURE = 0.6\nEVAL_MAX_NEW_TOKENS = 1024\n", "autoresearch/recipe.py": "\"\"\"THE ONE FILE AN EXPERIMENT MAY CHANGE: the SFT recipe (what the model is trained on, and how).\nEverything else (frozen eval, verifier, catalog, token budget, runner) is read-only. Keep it simple: all else equal,\nfewer rows / fewer knobs changed is better. `token_budget` is fixed by constants.py and must not be changed.\"\"\"\nfrom constants import FIXED_TOKEN_BUDGET\n\nRECIPE = {\n    \"token_budget\": FIXED_TOKEN_BUDGET,   # do not change\n    # ---- data mix ----\n    \"synth_train\": 8000,                  # generated flows (rows)\n    \"flow_repeat\": 15,                    # each real gold flow is repeated this many times\n    \"smoltalk_rows\": 3000,                # general chat rows so the model still talks (0 = none)\n    \"include_catalog\": True,              # catalog Q&A rows (\"which action ...\", \"what inputs ...\")\n    \"step_weights\": [25, 35, 25, 15],     # share of generated flows with 1, 2, 3, 4 steps\n    \"wrap_weights\": [55, 17, 14, 14],     # share with no container / loop / if / try\n    # ---- optimisation ----\n    \"seq_len\": 2048,\n    \"device_batch\": 4,\n    \"grad_accum\": 4,\n    \"init_lr_frac\": 0.8,                  # starting learning rate as a fraction of the pretrained base rate\n    \"warmdown_ratio\": 0.5,\n    \"warmup_ratio\": 0.0,\n    \"final_lr_frac\": 0.0,\n}\n", "autoresearch/loop.py": "\"\"\"Scripted autoresearch loop (no AI agent needed): greedy hill-climb over the SFT recipe, one change per experiment.\n\n baseline -> propose ONE mutation of the current best recipe -> run a fixed-budget experiment -> score on the frozen eval ->\n keep it only if it beats the best by more than KEEP_EPSILON, else discard -> log -> repeat.\nResumes from results.tsv (re-running skips experiments already done). `runner(recipe) -> dict` is injected, so the logic\nis tested without a GPU. results.tsv columns: n, score, real_score, solved_rate, val_bpb, minutes, status, description, recipe_json\n\"\"\"\nimport copy, csv, hashlib, json, os, random, time\nfrom constants import KEEP_EPSILON\n\nSEARCH_SPACE = {\n    \"init_lr_frac\":   [0.3, 0.5, 0.8, 1.2, 1.6],\n    \"warmdown_ratio\": [0.2, 0.5, 0.8],\n    \"warmup_ratio\":   [0.0, 0.05],\n    \"final_lr_frac\":  [0.0, 0.1],\n    \"synth_train\":    [4000, 8000, 16000, 30000],\n    \"flow_repeat\":    [5, 15, 30],\n    \"smoltalk_rows\":  [0, 1500, 3000],\n    \"include_catalog\": [True, False],\n    \"step_weights\":   [[25, 35, 25, 15], [10, 30, 30, 30], [40, 40, 15, 5], [0, 25, 35, 40]],\n    \"wrap_weights\":   [[55, 17, 14, 14], [30, 25, 20, 25], [80, 7, 7, 6]],\n}\n# Informed order: tried first (each costs ~9 minutes). Reasons come from the first results: a LOWER learning rate hurt, so try\n# HIGHER; real flows may be memorised through flow_repeat=15; dropping general chat rows leaves more budget for flows.\n# Deliberately NOT here: include_catalog=False (would likely raise the score by deleting catalog Q&A, which the frozen test does not measure).\nPRIORITY = [(\"init_lr_frac\", 1.2), (\"init_lr_frac\", 1.6), (\"flow_repeat\", 5), (\"smoltalk_rows\", 0),\n            (\"warmdown_ratio\", 0.8), (\"warmup_ratio\", 0.05), (\"flow_repeat\", 30)]\nCOLS = [\"n\", \"score\", \"real_score\", \"solved_rate\", \"val_bpb\", \"minutes\", \"status\", \"description\", \"recipe_json\"]\n\n\ndef rhash(recipe):\n    return hashlib.sha1(json.dumps(recipe, sort_keys=True).encode()).hexdigest()[:10]\n\n\ndef read_results(path):\n    if not os.path.exists(path):\n        return []\n    with open(path, newline=\"\", encoding=\"utf-8\") as f:\n        return list(csv.DictReader(f, delimiter=\"\\t\"))\n\n\ndef append_result(path, row):\n    new = not os.path.exists(path)\n    with open(path, \"a\", newline=\"\", encoding=\"utf-8\") as f:\n        w = csv.DictWriter(f, COLS, delimiter=\"\\t\")\n        if new:\n            w.writeheader()\n        w.writerow({k: row.get(k, \"\") for k in COLS})\n\n\ndef mutations(best, tried, rng):\n    out = []\n    for knob, values in SEARCH_SPACE.items():\n        for v in values:\n            if best.get(knob) == v:\n                continue\n            cand = copy.deepcopy(best)\n            cand[knob] = v\n            if rhash(cand) not in tried:\n                out.append((knob, v, cand))\n    rng.shuffle(out)\n    rank = {kv: i for i, kv in enumerate(PRIORITY)}\n    out.sort(key=lambda t: rank.get((t[0], t[1] if not isinstance(t[1], list) else tuple(t[1])), len(PRIORITY)))   # stable: priority first, rest stay shuffled\n    return out\n\n\ndef run_loop(runner, base_recipe, results_path, max_experiments=20, seed=0, on_event=print, on_keep=None):\n    rng = random.Random(seed)\n    rows = read_results(results_path)\n    tried, best, best_score = set(), copy.deepcopy(base_recipe), None\n    for r in rows:                                   # replay history so a resumed run continues where it stopped\n        rec = json.loads(r[\"recipe_json\"])\n        tried.add(rhash(rec))\n        if r[\"status\"] in (\"keep\", \"baseline\") and r[\"score\"]:\n            best, best_score = rec, float(r[\"score\"])\n    done = len(rows)\n    if best_score is None:                           # first run: the unchanged recipe is the baseline\n        res = runner(best)\n        append_result(results_path, {\"n\": done, \"score\": res[\"score\"], \"real_score\": res.get(\"real_score\", \"\"),\n                                     \"solved_rate\": res.get(\"solved_rate\", \"\"), \"val_bpb\": res.get(\"val_bpb\", \"\"),\n                                     \"minutes\": res.get(\"minutes\", \"\"), \"status\": \"baseline\", \"description\": \"baseline\",\n                                     \"recipe_json\": json.dumps(best, sort_keys=True)})\n        best_score, done = res[\"score\"], done + 1\n        tried.add(rhash(best))\n        if on_keep:\n            on_keep(best, best_score)\n        on_event(f\"baseline score {best_score:.4f}\")\n    while done < max_experiments:\n        cands = mutations(best, tried, rng)\n        if not cands:\n            on_event(\"search space exhausted\"); break\n        knob, value, cand = cands[0]\n        tried.add(rhash(cand))\n        desc = f\"{knob}: {best.get(knob)} -> {value}\"\n        t0 = time.time()\n        try:\n            res = runner(cand)\n            status = \"keep\" if res[\"score\"] > best_score + KEEP_EPSILON else \"discard\"\n        except Exception as ex:                      # OOM, bad config, ...: log and move on\n            res, status = {\"score\": 0.0}, \"crash\"\n            desc += f\" ({str(ex)[:80]})\"\n        append_result(results_path, {\"n\": done, \"score\": f\"{res['score']:.4f}\", \"real_score\": res.get(\"real_score\", \"\"),\n                                     \"solved_rate\": res.get(\"solved_rate\", \"\"), \"val_bpb\": res.get(\"val_bpb\", \"\"),\n                                     \"minutes\": res.get(\"minutes\", round((time.time() - t0) / 60, 1)), \"status\": status,\n                                     \"description\": desc, \"recipe_json\": json.dumps(cand, sort_keys=True)})\n        if status == \"keep\":\n            best, best_score = cand, res[\"score\"]\n            if on_keep:\n                on_keep(best, best_score)\n        on_event(f\"#{done} {desc} -> {res['score']:.4f} [{status}] (best {best_score:.4f})\")\n        done += 1\n    return best, best_score\n", "autoresearch/run_experiment.py": "\"\"\"Run ONE fixed-budget experiment: build data from recipe.py -> SFT for exactly token_budget tokens -> frozen eval -> result.\nREAD-ONLY for agents (like Karpathy's prepare.py). Usage:\n  python run_experiment.py --platform PLATFORM_REPO --catalog CATALOG.jsonl --workdir DIR --nanochat NANOCHAT_DIR \\\n      --base-dir NANOCHAT_BASE_DIR [--recipe recipe.py] [--dry-run]\nPrints a summary block (grep-able) and writes WORKDIR/result.json. --dry-run builds data, checks leakage and prints the\ncommands without training (works on CPU).\"\"\"\nimport argparse, json, math, os, re, runpy, subprocess, sys, time\nfrom pathlib import Path\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE)); sys.path.insert(0, str(HERE.parent))\nfrom constants import FIXED_TOKEN_BUDGET\nimport gen_flows as G\n\nALLOWED = {\"token_budget\", \"synth_train\", \"flow_repeat\", \"smoltalk_rows\", \"include_catalog\", \"step_weights\", \"wrap_weights\",\n           \"seq_len\", \"device_batch\", \"grad_accum\", \"init_lr_frac\", \"warmdown_ratio\", \"warmup_ratio\", \"final_lr_frac\"}\n\nap = argparse.ArgumentParser()\nap.add_argument(\"--platform\", required=True); ap.add_argument(\"--catalog\", required=True)\nap.add_argument(\"--workdir\", required=True); ap.add_argument(\"--nanochat\", default=\"/kaggle/working/nanochat\")\nap.add_argument(\"--base-dir\", default=os.environ.get(\"NANOCHAT_BASE_DIR\", \"\"))\nap.add_argument(\"--recipe\", default=str(HERE / \"recipe.py\")); ap.add_argument(\"--frozen-dir\", default=str(HERE / \"frozen\"))\nap.add_argument(\"--ooa-dir\", default=str(HERE.parent)); ap.add_argument(\"--dry-run\", action=\"store_true\")\na = ap.parse_args()\n\n\ndef load_recipe(path):\n    r = runpy.run_path(path)[\"RECIPE\"]\n    extra = set(r) - ALLOWED\n    assert not extra, f\"unknown recipe keys: {sorted(extra)}\"\n    assert r[\"token_budget\"] == FIXED_TOKEN_BUDGET, \"token_budget is fixed (constants.py); experiments must stay comparable\"\n    assert r[\"seq_len\"] in (2048, 4096), \"seq_len must be 2048 or 4096: real flows are longer than 1024 tokens and break nanochat's packing at 1024 (experiment #5: val_bpb 9.0)\" and r[\"device_batch\"] >= 1 and r[\"grad_accum\"] >= 1\n    assert len(r[\"step_weights\"]) == 4 and len(r[\"wrap_weights\"]) == 4 and sum(r[\"step_weights\"]) > 0 and sum(r[\"wrap_weights\"]) > 0\n    assert 0 < r[\"init_lr_frac\"] <= 3 and 0 <= r[\"warmdown_ratio\"] <= 1\n    return r\n\n\ndef read(path):\n    return [json.loads(l) for l in open(path, encoding=\"utf-8\")] if os.path.exists(path) else []\n\n\ndef write(path, rows):\n    with open(path, \"w\", encoding=\"utf-8\") as f:\n        for r in rows:\n            f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n\n\nR = load_recipe(a.recipe)\nwork = Path(a.workdir); sft = work / \"sft\"; sft.mkdir(parents=True, exist_ok=True)\nt_start = time.time()\n\n# ---- 1. data ---------------------------------------------------------------------------------------------------------\nsubprocess.run([sys.executable, str(a.ooa_dir + \"/build_sft_data.py\"), a.platform, str(sft), \"9000\"], check=True,\n               stdout=subprocess.DEVNULL)\ncatalog = read(sft / \"catalog.jsonl\")\nsynth = [{\"messages\": [{\"role\": \"user\", \"content\": q}, {\"role\": \"assistant\", \"content\": m}]}\n         for q, _, m in G.generate(catalog, a.platform, R[\"synth_train\"], 1, cfg=R)]\nwrite(sft / \"synth_train.jsonl\", synth)\n# the recipe may not train on the frozen eval: drop any row whose request or answer appears in it\nfrozen = read(Path(a.frozen_dir) / \"frozen_eval.jsonl\")\nbad_prompts = {r[\"prompt\"] for r in frozen}; bad_flows = {r[\"gold_flow\"] for r in frozen}\ndropped = 0\nfor name in (\"train_flows\", \"train_catalog\", \"synth_train\", \"train\"):\n    rows = read(sft / f\"{name}.jsonl\")\n    keep = [r for r in rows if r[\"messages\"][0][\"content\"] not in bad_prompts and r[\"messages\"][1][\"content\"] not in bad_flows]\n    dropped += len(rows) - len(keep)\n    write(sft / f\"{name}.jsonl\", keep)\n# unseen validation rows: held-out catalog Q&A only\nval = [r for r in read(sft / \"val.jsonl\") if r[\"messages\"][1][\"content\"].startswith((\"Use \", \"`\"))]\nwrite(sft / \"val.jsonl\", val)\n\n# ---- 2. how many passes does the token budget need? (estimate: 3.3 characters per token) --------------------------------\ndef est_tokens(rows):\n    return sum(len(r[\"messages\"][0][\"content\"]) + len(r[\"messages\"][1][\"content\"]) for r in rows) / 3.3\n\nflows, cats, syn = read(sft / \"train_flows.jsonl\"), read(sft / \"train_catalog.jsonl\"), read(sft / \"synth_train.jsonl\")\none_pass = est_tokens(flows) * R[\"flow_repeat\"] + (est_tokens(cats) if R[\"include_catalog\"] else 0) + est_tokens(syn) + R[\"smoltalk_rows\"] * 330\nepochs = max(1, min(30, math.ceil(R[\"token_budget\"] / max(1, one_pass))))\ntokens_per_step = R[\"seq_len\"] * R[\"device_batch\"] * R[\"grad_accum\"]\niters = R[\"token_budget\"] // tokens_per_step\nopt_steps = -(-iters // R[\"grad_accum\"])                     # nanochat counts --num-iterations in MICRO-batches (grad_accum per optimizer step)\nreal_tokens = iters * R[\"seq_len\"] * R[\"device_batch\"]      # what is really trained on (about budget / grad_accum)\nprint(f\"data: flows {len(flows)}x{R['flow_repeat']} catalog {len(cats) if R['include_catalog'] else 0} synth {len(syn)} \"\n      f\"smoltalk {R['smoltalk_rows']} | leaked rows dropped: {dropped} | ~{one_pass/1e6:.1f}M tokens in the mixture \"\n      f\"| trains {iters} micro-batches = {opt_steps} optimizer steps = {real_tokens/1e6:.1f}M tokens (a slice of the mixture)\")\n\nsft_cmd = [sys.executable, \"-m\", \"scripts.chat_sft\", f\"--max-seq-len={R['seq_len']}\", f\"--device-batch-size={R['device_batch']}\",\n           f\"--total-batch-size={tokens_per_step}\", f\"--num-iterations={iters}\", f\"--init-lr-frac={R['init_lr_frac']}\",\n           f\"--warmdown-ratio={R['warmdown_ratio']}\", f\"--warmup-ratio={R['warmup_ratio']}\", f\"--final-lr-frac={R['final_lr_frac']}\",\n           \"--eval-every=-1\", \"--chatcore-every=-1\", f\"--eval-tokens={tokens_per_step * 2}\", \"--run=dummy\"]\neval_cmd = [sys.executable, \"-m\", \"scripts.ooa_eval_frozen\", f\"--frozen-dir={a.frozen_dir}\", f\"--out={work / 'eval.json'}\",\n            f\"--platform={a.platform}\", f\"--ooa-dir={a.ooa_dir}\"]\nif a.dry_run:\n    print(\"DRY RUN, nothing trained.\\n  sft :\", \" \".join(sft_cmd[1:]), \"\\n  eval:\", \" \".join(eval_cmd[1:]))\n    sys.exit(0)\n\n# ---- 3. patch nanochat's SFT script to use this recipe, train, evaluate --------------------------------------------------\nnc = Path(a.nanochat)\nsubprocess.run([\"git\", \"checkout\", \"-q\", \"--\", \"scripts/chat_sft.py\"], cwd=nc, check=True)\nsubprocess.run([\"cp\", str(Path(a.ooa_dir) / \"ooa_task.py\"), str(nc / \"tasks/ooa.py\")], check=True)\nsubprocess.run([\"cp\", str(HERE / \"ooa_eval_frozen.py\"), str(nc / \"scripts/ooa_eval_frozen.py\")], check=True)\np = nc / \"scripts/chat_sft.py\"; s = p.read_text()\ns = s.replace(\"from tasks.smoltalk import SmolTalk\", \"from tasks.smoltalk import SmolTalk\\nfrom tasks.ooa import OOADialogues\")\ni = s.index(\"train_tasks = [\"); j = s.index(\"\\n\", s.index(\"# total: 24K\"))\nparts = [f'OOADialogues(\"{sft}/train_flows.jsonl\")' for _ in range(R[\"flow_repeat\"])]\nif R[\"include_catalog\"]:\n    parts.append(f'OOADialogues(\"{sft}/train_catalog.jsonl\")')\nparts.append(f'OOADialogues(\"{sft}/synth_train.jsonl\")')\nnew = (f\"train_tasks = [{', '.join(parts)}] * {epochs}\\n\"\n       + (f'train_tasks.append(SmolTalk(split=\"train\", stop={R[\"smoltalk_rows\"]}))\\n' if R[\"smoltalk_rows\"] > 0 else \"\")\n       + f'train_dataset = TaskMixture(train_tasks)\\nval_dataset = TaskMixture([OOADialogues(\"{sft}/val.jsonl\")])')\np.write_text(s[:i] + new + s[j:])\nenv = {**os.environ, \"NANOCHAT_BASE_DIR\": a.base_dir}\nimport shutil\n_ck, _bk = Path(a.base_dir) / \"chatsft_checkpoints\", Path(a.base_dir) / \"chatsft_checkpoints_before_autoresearch\"\nif _ck.exists() and not _bk.exists():\n    shutil.move(str(_ck), str(_bk))          # keep your earlier SFT model: it is moved aside once, never deleted\nelse:\n    shutil.rmtree(_ck, ignore_errors=True)   # later experiments overwrite only experiment output\nt0 = time.time()\nlog = subprocess.run(sft_cmd, cwd=nc, env=env, capture_output=True, text=True)\n(work / \"sft.log\").write_text(log.stdout + log.stderr)\nif log.returncode != 0:\n    print((log.stdout + log.stderr)[-1500:]); sys.exit(1)\nbpb = re.findall(r\"Minimum validation bpb: ([\\d.]+)\", log.stdout)\nev = subprocess.run(eval_cmd, cwd=nc, env=env, capture_output=True, text=True)\n(work / \"eval.log\").write_text(ev.stdout + ev.stderr)\nif ev.returncode != 0:\n    print((ev.stdout + ev.stderr)[-1500:]); sys.exit(1)\nres = json.load(open(work / \"eval.json\"))\nif bpb and float(bpb[-1]) > 5:\n    print(\"TRAINING FAILED: validation loss\", bpb[-1], \"(a trained model is below 1). Not counted as a result.\")\n    sys.exit(1)\nres.update({\"val_bpb\": float(bpb[-1]) if bpb else None, \"minutes\": round((time.time() - t_start) / 60, 1),\n            \"train_tokens_M\": round(real_tokens / 1e6, 1), \"optimizer_steps\": opt_steps, \"leaked_rows_dropped\": dropped})\njson.dump(res, open(work / \"result.json\", \"w\"))\nprint(\"---\")\nfor k in (\"score\", \"real_score\", \"solved_rate\", \"valid_rate\", \"val_bpb\", \"minutes\", \"train_tokens_M\"):\n    print(f\"{k + ':':16s}{res.get(k)}\")\n", "autoresearch/ooa_eval_frozen.py": "\"\"\"Frozen scoreboard for OOA experiments (copy to nanochat/scripts/). NOT to be edited by experiments.\nChecks the frozen set's hash, samples EVAL_SAMPLES answers per request from the freshly trained SFT model, and scores each\nwith the verifier against the FROZEN catalog snapshot. score = mean reward on the generated requests (smooth signal);\nreal_score = mean reward on the real held-out flows (reality check: long, hard); solved_rate / valid_rate over generated ones.\"\"\"\nimport argparse, hashlib, json, os, sys\nfrom nanochat.common import compute_init, autodetect_device_type\nfrom nanochat.checkpoint_manager import load_model\nfrom nanochat.engine import Engine\n\nap = argparse.ArgumentParser()\nap.add_argument(\"--frozen-dir\", required=True); ap.add_argument(\"--out\", required=True)\nap.add_argument(\"--platform\", required=True); ap.add_argument(\"--ooa-dir\", required=True)\nargs = ap.parse_args()\nsys.path.insert(0, args.ooa_dir); sys.path.insert(0, os.path.join(args.ooa_dir, \"autoresearch\"))\nfrom constants import EVAL_SAMPLES, EVAL_TEMPERATURE, EVAL_MAX_NEW_TOKENS\n\nbody = open(os.path.join(args.frozen_dir, \"frozen_eval.jsonl\"), encoding=\"utf-8\").read()\nwant = open(os.path.join(args.frozen_dir, \"frozen_eval.sha256\")).read().strip()\nassert hashlib.sha256(body.encode()).hexdigest() == want, \"frozen eval was modified: results would not be comparable\"\nos.environ[\"OOA_CATALOG\"] = os.path.join(args.frozen_dir, \"frozen_catalog.jsonl\")\nos.environ[\"OOA_SCHEMA\"] = os.path.join(args.platform, \"contracts/flow-schema/flow.v1.json\")\nimport ooa_reward as R\n\nrows = [json.loads(l) for l in body.splitlines() if l.strip()]\n_, _, _, _, device = compute_init(autodetect_device_type())\nmodel, tok, _ = load_model(\"sft\", device, phase=\"eval\")\nengine = Engine(model, tok); model.eval()\nagg = {\"synthetic\": [], \"real\": []}\nfor i, r in enumerate(rows):\n    conv = {\"messages\": [{\"role\": \"user\", \"content\": r[\"prompt\"]}, {\"role\": \"assistant\", \"content\": r[\"gold_flow\"]}]}\n    tokens = tok.render_for_completion(conv)\n    seqs, _ = engine.generate_batch(tokens, num_samples=EVAL_SAMPLES, max_tokens=EVAL_MAX_NEW_TOKENS,\n                                    temperature=EVAL_TEMPERATURE, top_k=50, seed=4242 + i)   # fixed seeds: same questions, same dice\n    for s in seqs:\n        sc = R.score(R.clean(tok.decode(s[len(tokens):])), r[\"gold_actions\"])\n        agg[r[\"kind\"]].append(sc)\nmean = lambda xs, k: round(sum(x[k] for x in xs) / max(1, len(xs)), 4)\nres = {\"score\": mean(agg[\"synthetic\"], \"total\"), \"real_score\": mean(agg[\"real\"], \"total\"),\n       \"solved_rate\": round(sum(x[\"solved\"] for x in agg[\"synthetic\"]) / max(1, len(agg[\"synthetic\"])), 4),\n       \"valid_rate\": round(sum(x[\"valid\"] for x in agg[\"synthetic\"]) / max(1, len(agg[\"synthetic\"])), 4),\n       \"parts\": {k: mean(agg[\"synthetic\"], k) for k in R.W}}\njson.dump(res, open(args.out, \"w\"))\nprint(json.dumps(res))\n", "retrieval.py": "\"\"\"Action retrieval + prompt format. Pure standard library (BM25 over the catalog), works anywhere, also in a browser worker.\n\nWhy: the model invents action and input names when it has to recall ~1,300 of them. Instead each request carries the\nmost likely candidate actions (id, required inputs, optional inputs), so the model copies names instead of remembering them.\n\n    idx = ActionIndex(catalog)            # catalog = list of entries from catalog.jsonl\n    idx.search(\"read invoices from csv\", k=10)\n    make_prompt(request, idx.search(request, 10))  ->  text the model is trained and run on\n\"\"\"\nimport math, re\nfrom collections import Counter\n\nSTOP = set(\"a an the and or of to in on for with from by is are be it its this that as at into then than each every all any \"\n           \"me my your our their can will should would please make create build flow automation automations do does\".split())\n\n\ndef tokens(text):\n    out = []\n    for w in re.split(r\"[^A-Za-z0-9]+\", text.lower()):\n        if not w or w in STOP:\n            continue\n        out.append(w)\n        if w.endswith(\"s\") and len(w) > 3:      # crude plural folding: invoices -> invoice\n            out.append(w[:-1])\n    return out\n\n\ndef _doc_tokens(e):\n    t = tokens(e[\"id\"]) * 2 + tokens(e[\"name\"]) * 2 + tokens(e[\"toolkit\"]) + tokens(e[\"description\"])\n    t += tokens(\" \".join(i[\"name\"] for i in e[\"inputs\"]))\n    t += tokens(\" \".join(o[\"name\"] for o in e[\"outputs\"]))\n    return t\n\n\nclass ActionIndex:\n    def __init__(self, catalog, kinds=(\"action\",), k1=1.4, b=0.75):\n        self.entries = [e for e in catalog if e[\"kind\"] in kinds]\n        self.k1, self.b = k1, b\n        self.docs = [Counter(_doc_tokens(e)) for e in self.entries]\n        self.len = [sum(d.values()) for d in self.docs]\n        self.avg = sum(self.len) / max(1, len(self.len))\n        df = Counter()\n        for d in self.docs:\n            df.update(d.keys())\n        n = len(self.docs)\n        self.idf = {w: math.log(1 + (n - c + 0.5) / (c + 0.5)) for w, c in df.items()}\n        self.inv = {}\n        for i, d in enumerate(self.docs):\n            for w in d:\n                self.inv.setdefault(w, []).append(i)\n        self.by_id = {e[\"id\"]: e for e in self.entries}\n\n    def search(self, query, k=10):\n        q = Counter(tokens(query))\n        scores = {}\n        for w in q:\n            if w not in self.inv:\n                continue\n            for i in self.inv[w]:\n                f = self.docs[i][w]\n                s = self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))\n                scores[i] = scores.get(i, 0.0) + s\n        top = sorted(scores, key=lambda i: -scores[i])[:k]\n        return [self.entries[i] for i in top]\n\n\ndef candidate_line(e):\n    req = [i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]]\n    opt = [i[\"name\"] for i in e[\"inputs\"] if not i[\"required\"]][:6]\n    outs = [o[\"name\"] for o in e[\"outputs\"]][:3]\n    s = f\"{e['id']} | in: {','.join(req) or '-'}\"\n    if opt:\n        s += f\" | opt: {','.join(opt)}\"\n    if outs:\n        s += f\" | out: {','.join(outs)}\"\n    return s\n\n\ndef make_prompt(request, candidates):\n    lines = \"\\n\".join(candidate_line(e) for e in candidates)\n    return f\"Actions:\\n{lines}\\n\\nRequest: {request}\"\n\n\ndef with_gold(candidates, gold_ids, by_id, k):\n    \"\"\"Training only: make sure the gold actions are in the candidate list (replace the lowest-ranked extras).\"\"\"\n    ids = [e[\"id\"] for e in candidates]\n    need = [g for g in dict.fromkeys(gold_ids) if g not in ids and g in by_id]\n    keep = candidates[: max(0, k - len(need))]\n    return keep + [by_id[g] for g in need]\n", "repair.py": "\"\"\"Deterministic repair of near-miss flows before verification: snap misspelled action ids and input/output names to\nthe closest real one (only when very close), so small typos do not cost an escalation. Never invents anything new and\nnever adds or removes steps. Returns (flow, notes).\"\"\"\nimport copy, difflib, re\n\n\ndef repair(flow, cat):\n    \"\"\"cat: dict action_id -> catalog entry.\"\"\"\n    flow = copy.deepcopy(flow)\n    notes, ids = [], list(cat)\n    def walk(nodes):\n        for n in nodes or []:\n            if isinstance(n, dict):\n                yield n\n                for k in (\"children\", \"else\", \"finally\"):\n                    if isinstance(n.get(k), list):\n                        yield from walk(n[k])\n                for c in n.get(\"catch\") or []:\n                    if isinstance(c, dict):\n                        yield from walk(c.get(\"children\"))\n    for s in walk(flow.get(\"steps\")):\n        a = s.get(\"action\")\n        if isinstance(a, str) and a not in cat:\n            m = difflib.get_close_matches(a, ids, n=1, cutoff=0.88)\n            if m:\n                notes.append(f\"action {a} -> {m[0]}\")\n                s[\"action\"] = a = m[0]\n        e = cat.get(a)\n        if not e or a.startswith(\"core/\"):\n            continue\n        for key, valid in ((\"args\", [i[\"name\"] for i in e[\"inputs\"]]), (\"out\", [o[\"name\"] for o in e[\"outputs\"]])):\n            d = s.get(key)\n            if not isinstance(d, dict):\n                continue\n            for k in list(d):\n                if k not in valid:\n                    m = difflib.get_close_matches(k, valid, n=1, cutoff=0.8)\n                    if not m:\n                        m = _contained_match(k, valid, d, e, key)\n                    if m and m[0] not in d:\n                        notes.append(f\"{a}.{key} {k} -> {m[0]}\")\n                        d[m[0]] = d.pop(k)\n    _fix_ids(flow, notes)\n    _declare_used_inputs(flow, notes, walk)\n    return flow, notes\n\n\ndef _contained_match(k, valid, d, e, key):\n    \"\"\"A wrong key that clearly means one real name: for inputs, the only missing REQUIRED input whose name is contained\n    in the key or contains it (\"target_folder\" -> \"target\"); for outputs, the action's only output.\"\"\"\n    if key == \"out\":\n        return valid if len(valid) == 1 else None\n    missing = [i[\"name\"] for i in e[\"inputs\"] if i[\"required\"] and i[\"name\"] not in d]\n    hits = [n for n in missing if n in k or k in n]\n    return hits if len(hits) == 1 else None\n\n\n_ID = re.compile(r\"[^A-Za-z0-9_.-]+\")\n\n\ndef _fix_ids(flow, notes):\n    \"\"\"Flow id must match ^[A-Za-z0-9_.-]{1,64}$ (schema); step ids must be unique.\"\"\"\n    fid = flow.get(\"id\")\n    if isinstance(fid, str):\n        good = _ID.sub(\"_\", fid.strip()).strip(\"_\")[:64] or \"flow\"\n        if good != fid:\n            notes.append(f\"flow id {fid!r} -> {good!r}\")\n            flow[\"id\"] = good\n\n\ndef _declare_used_inputs(flow, notes, walk):\n    \"\"\"A step reads ${name} / $name but the flow never declares it: declare it as a String input the user fills in.\n    Only for plain names that are not built in (sys, global) and not written by any step's out.\"\"\"\n    import ooa_reward as R\n    variables = flow.get(\"variables\")\n    if not isinstance(variables, list):\n        return\n    declared = {v.get(\"name\") for v in variables if isinstance(v, dict)}\n    steps = list(walk(flow.get(\"steps\")))\n    written = {v for s in steps if isinstance(s.get(\"out\"), dict) for v in s[\"out\"].values() if isinstance(v, str)}\n    loop_vars = {s[\"args\"].get(\"as\") for s in steps if isinstance(s.get(\"args\"), dict) and isinstance(s[\"args\"].get(\"as\"), str)}\n    ids = [s.get(\"id\") for s in steps]\n    if len(ids) != len(set(ids)):\n        for k, s in enumerate(steps, 1):\n            s[\"id\"] = f\"s{k}\"\n        notes.append(\"step ids renumbered (they were not unique)\")\n    for name in sorted((written | loop_vars) - declared):      # a step stores into it, or a loop names its item\n        variables.append({\"name\": name, \"type\": \"Any\", \"direction\": \"local\"})\n        declared.add(name)\n        notes.append(f\"declared missing local variable {name!r}\")\n    for name in sorted({r for s in steps if isinstance(s.get(\"args\"), dict) for r in R.refs(s[\"args\"])}):\n        if name in declared:\n            continue\n        variables.append({\"name\": name, \"type\": \"String\", \"direction\": \"input\", \"required\": True,\n                          \"description\": \"Added by repair: the flow reads this value but did not declare it.\"})\n        declared.add(name)\n        notes.append(f\"declared missing input variable {name!r}\")\n", "serving/ooa_server.py": "\"\"\"OOA assistant server: one OpenAI-compatible endpoint, a router, a verifier, and client data-policy switches.\n\n   request -> [mask PII for cloud] -> router\n        simple?  -> LOCAL model (our own) -> repair -> verify -> ok? return\n        else / failed -> CLOUD model (Claude) if the client policy allows -> repair -> verify (one retry with the errors)\n        nothing verified -> honest error (never returns an unchecked flow as if it were fine)\n   POST /v1/chat/completions   OpenAI-style; the last user message is the request; the answer is the flow JSON\n   POST /v1/feedback           {\"prompt\",\"flow\",\"accepted\",\"edited_flow\"}  -> feedback.jsonl (only if the policy allows)\n   GET  /health\n\nRun:  python ooa_server.py --catalog catalog.jsonl --platform PLATFORM_REPO --policy policy.json [--local nanochat] [--port 8080]\nLocal backend: any object with generate(request, attempt) -> str (see backends.py). Cloud: Anthropic Messages API (env ANTHROPIC_API_KEY).\nDefault policy is LOCAL-ONLY and stores nothing.\n\"\"\"\nimport argparse, json, os, re, sys, time, threading\nfrom dataclasses import dataclass, asdict\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE.parent / \"training\"))\nimport ooa_reward as R\nimport retrieval\nimport repair as RP\n\n\n@dataclass\nclass Policy:\n    allow_escalation: bool = False       # may a request leave for the cloud model at all\n    mask_pii: bool = True                # mask emails, phones, card-like numbers before anything leaves\n    store_logs: bool = False             # keep a log of requests and results (masked)\n    allow_training_on_data: bool = False  # may logged data / feedback be used to train models\n    local_retries: int = 2\n    max_clauses_local: int = 6           # longer / more complex requests go to the cloud straight away\n    cloud_model: str = \"claude-sonnet-5-5\"\n    cloud_max_tokens: int = 6000\n    log_dir: str = \"ooa_logs\"\n\n    @classmethod\n    def load(cls, path=None):\n        d = json.load(open(path)) if path else {}\n        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})\n\n\nPII = [(re.compile(r\"[\\w.+-]+@[\\w-]+\\.[\\w.-]+\"), \"[EMAIL]\"),\n       (re.compile(r\"\\b(?:\\d[ -]?){13,19}\\b\"), \"[CARD_OR_ID_NUMBER]\"),\n       (re.compile(r\"(?<!\\w)\\+?\\d[\\d ()-]{8,}\\d\"), \"[PHONE]\"),\n       (re.compile(r\"\\b\\d{1,3}(?:\\.\\d{1,3}){3}\\b\"), \"[IP]\")]\n\n\ndef mask(text):\n    for rx, rep in PII:\n        text = rx.sub(rep, text)\n    return text\n\n\ndef clauses(text):\n    parts = re.split(r\",|;|\\bthen\\b|\\bafter that\\b|\\band\\b\", text, flags=re.I)\n    return len([p for p in parts if len(p.split()) >= 2])\n\n\ndef extract_json(text):\n    text = R.clean(text).strip()\n    text = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", text).strip()\n    i, j = text.find(\"{\"), text.rfind(\"}\")\n    return text[i:j + 1] if i != -1 and j > i else text\n\n\ndef parse_messages(messages):\n    \"\"\"OpenAI-style messages -> (request, system, assistant_mode, existing_variables, full_user_text).\n    The platform's Studio assistant sends: system = long instructions + its action catalog, user = 'Existing variables: [...]\\nRequest: ...'\n    and reads back {\"variables\": [...], \"steps\": [...], \"notes\": \"...\"}. Anything else is treated as a plain request.\"\"\"\n    system = \"\\n\".join(m[\"content\"] for m in messages if m.get(\"role\") == \"system\" and isinstance(m.get(\"content\"), str))\n    users = [m for m in messages if m.get(\"role\") == \"user\"]\n    text = users[-1].get(\"content\", \"\") if users else \"\"\n    if isinstance(text, list):\n        text = \" \".join(x.get(\"text\", \"\") for x in text if isinstance(x, dict))\n    assistant = text.startswith(\"Existing variables:\") and \"\\nRequest:\" in text\n    existing = []\n    if assistant:\n        try:\n            existing = json.loads(text[len(\"Existing variables:\"):text.index(\"\\nRequest:\")])\n            existing = existing if isinstance(existing, list) else []\n        except Exception:\n            existing = []\n        request = text.split(\"\\nRequest:\", 1)[1].strip()\n    else:\n        request = text\n    return request, system, assistant, existing, text\n\n\nclass Router:\n    def __init__(self, catalog, platform, policy, local=None, cloud=None):\n        os.environ[\"OOA_CATALOG\"] = catalog\n        os.environ[\"OOA_SCHEMA\"] = str(Path(platform) / \"contracts/flow-schema/flow.v1.json\")\n        R.configure()\n        self.cat = {}\n        entries = [json.loads(l) for l in open(catalog, encoding=\"utf-8\")]\n        for e in entries:\n            self.cat[e[\"id\"]] = e\n        self.index = retrieval.ActionIndex(entries)\n        rules = Path(platform) / \"docs/llms.txt\"\n        self.rules = rules.read_text(encoding=\"utf-8\") if rules.exists() else \"Write an OOA Flow v1 JSON document.\"\n        self.policy, self.local, self.cloud = policy, local, cloud\n        self.lock = threading.Lock()\n\n    # -- one attempt: parse -> repair -> verify ---------------------------------------------------\n    def check(self, raw, existing=None, assistant=False):\n        text = extract_json(raw)\n        try:\n            obj = json.loads(text)\n        except Exception:\n            return None, R.problems(text)\n        if not isinstance(obj, dict):\n            return None, [\"The answer is not a JSON object.\"]\n        # Studio's assistant format has no kind/id/name: wrap it so the same verifier applies\n        flow = obj if \"kind\" in obj else {\"kind\": \"TaskBot\", \"id\": \"assistant_check\", \"name\": \"assistant check\",\n                                          \"variables\": list(obj.get(\"variables\", [])), \"steps\": obj.get(\"steps\", [])}\n        names = {v.get(\"name\") for v in flow.get(\"variables\", []) if isinstance(v, dict)}\n        extra = [v for v in (existing or []) if isinstance(v, dict) and v.get(\"name\") not in names]\n        if extra:\n            flow = {**flow, \"variables\": list(flow.get(\"variables\", [])) + extra}   # variables already in the user's open agent\n        flow, notes = RP.repair(flow, self.cat)\n        mini = json.dumps(flow, separators=(\",\", \":\"), ensure_ascii=False)\n        problems = R.problems(mini)\n        if problems:\n            return None, problems\n        if assistant:\n            have = {v.get(\"name\") for v in (existing or []) if isinstance(v, dict)}\n            out = {\"variables\": [v for v in flow.get(\"variables\", []) if v.get(\"name\") not in have], \"steps\": flow[\"steps\"],\n                   \"notes\": \"Written by the OOA assistant and checked against the platform's rules.\"}\n            return json.dumps(out, separators=(\",\", \":\"), ensure_ascii=False), notes\n        return mini, notes\n\n    def cloud_prompt(self, request, errors):\n        cands = self.index.search(request, 40)\n        # whole toolkits of the best matches: descriptions rarely name low-level actions (click, type, open ...)\n        top_tk = []\n        for e in self.index.search(request, 200):\n            if e[\"toolkit_id\"] not in top_tk:\n                top_tk.append(e[\"toolkit_id\"])\n            if len(top_tk) == 4:\n                break\n        lines = {e[\"id\"]: retrieval.candidate_line(e) for e in cands}\n        for e in self.index.entries:\n            if e[\"toolkit_id\"] in top_tk and len(lines) < 160:\n                lines.setdefault(e[\"id\"], retrieval.candidate_line(e))\n        for cid in (\"core/if\", \"core/loop\", \"core/try\", \"core/set\", \"core/log\"):\n            if cid in self.cat:\n                lines.setdefault(cid, retrieval.candidate_line(self.cat[cid]))\n        system = (self.rules + \"\\n\\nReturn ONLY one minified OOA Flow JSON object, no prose, no code fences. \"\n                  \"Use only action ids from the list below, with exactly the input names shown (in: required, opt: optional).\\n\"\n                  \"Format of each line: action id | in: required inputs | opt: optional inputs | out: outputs\\n\"\n                  + \"\\n\".join(lines.values()))\n        user = request if not errors else request + \"\\n\\nYour previous answer had these problems, fix them:\\n- \" + \"\\n- \".join(errors)\n        return system, user\n\n    def run(self, request, system=None, assistant=False, existing=None, full_user=None):\n        t0, tried, errors = time.time(), [], []\n        p = self.policy\n        simple = clauses(request) <= p.max_clauses_local and len(request) <= 500\n        if self.local is not None and simple:\n            for attempt in range(1 + p.local_retries):\n                try:\n                    raw = self.local.generate(request, attempt)\n                except Exception as ex:\n                    errors = [f\"local model error: {ex}\"]\n                    break\n                flow, info = self.check(raw, existing, assistant)\n                tried.append({\"source\": \"local\", \"ok\": flow is not None})\n                if flow:\n                    return self.done(request, flow, \"local\", tried, info, t0)\n                errors = info\n        elif self.local is not None:\n            errors = [\"request looks complex; skipped the local model\"]\n        if self.cloud is not None and p.allow_escalation:\n            req = mask(request) if p.mask_pii else request\n            err = [mask(e) if p.mask_pii else e for e in errors if not e.startswith(\"request looks\")]\n            for attempt in range(2):\n                if assistant and system:      # the platform's own prompt carries its live action catalog: use it\n                    sys_p = system\n                    user = (mask(full_user) if p.mask_pii else full_user) + (\"\\n\\nYour previous answer had these problems, fix them:\\n- \" + \"\\n- \".join(err) if err else \"\")\n                else:\n                    sys_p, user = self.cloud_prompt(req, err)\n                try:\n                    raw = self.cloud.generate(sys_p, user)\n                except Exception as ex:\n                    err = [f\"cloud call failed: {ex}\"]\n                    tried.append({\"source\": \"cloud\", \"ok\": False, \"error\": str(ex)[:120]})\n                    break\n                flow, info = self.check(raw, existing, assistant)\n                tried.append({\"source\": \"cloud\", \"ok\": flow is not None})\n                if flow:\n                    return self.done(request, flow, \"cloud\", tried, info, t0)\n                err = info\n            errors = err\n        reason = (\"no model produced a flow that passed verification\" if (self.cloud and p.allow_escalation)\n                  else \"the local model could not produce a verified flow, and cloud escalation is disabled by this client's policy\")\n        return self.done(request, None, \"none\", tried, errors or [reason], t0, reason=reason)\n\n    def done(self, request, flow, source, tried, info, t0, reason=None):\n        res = {\"flow\": flow, \"source\": source, \"verified\": flow is not None, \"attempts\": tried,\n               \"repairs\" if flow else \"problems\": info, \"seconds\": round(time.time() - t0, 3)}\n        if reason:\n            res[\"reason\"] = reason\n        if self.policy.store_logs:\n            self.log(\"requests.jsonl\", {\"time\": time.strftime(\"%Y-%m-%dT%H:%M:%S\"), \"request\": mask(request),\n                                        \"source\": source, \"verified\": res[\"verified\"], \"flow\": flow,\n                                        \"training_ok\": self.policy.allow_training_on_data})\n        return res\n\n    def log(self, name, row):\n        d = Path(self.policy.log_dir)\n        d.mkdir(parents=True, exist_ok=True)\n        with self.lock, open(d / name, \"a\", encoding=\"utf-8\") as f:\n            f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n    def feedback(self, row):\n        if not self.policy.store_logs:\n            return {\"stored\": False, \"reason\": \"policy: store_logs is off\"}\n        row = {**row, \"prompt\": mask(row.get(\"prompt\", \"\")), \"training_ok\": self.policy.allow_training_on_data,\n               \"time\": time.strftime(\"%Y-%m-%dT%H:%M:%S\")}\n        self.log(\"feedback.jsonl\", row)\n        return {\"stored\": True, \"usable_for_training\": self.policy.allow_training_on_data}\n\n\nclass CloudBackend:\n    \"\"\"Claude through the Anthropic Messages API (needs ANTHROPIC_API_KEY). In production route this through the\n    platform's AI Gateway instead (it already adds PII masking, budgets and the governance log).\"\"\"\n    def __init__(self, model, max_tokens=6000, api_key=None, url=\"https://api.anthropic.com/v1/messages\"):\n        self.model, self.max_tokens, self.url = model, max_tokens, url\n        self.key = api_key or os.environ.get(\"ANTHROPIC_API_KEY\")\n\n    def generate(self, system, user):\n        import urllib.request\n        if not self.key:\n            raise RuntimeError(\"ANTHROPIC_API_KEY is not set\")\n        body = json.dumps({\"model\": self.model, \"max_tokens\": self.max_tokens, \"system\": system,\n                           \"messages\": [{\"role\": \"user\", \"content\": user}]}).encode()\n        req = urllib.request.Request(self.url, body, {\"x-api-key\": self.key, \"anthropic-version\": \"2023-06-01\",\n                                                      \"content-type\": \"application/json\"})\n        with urllib.request.urlopen(req, timeout=120) as r:\n            data = json.loads(r.read())\n        return \"\".join(b.get(\"text\", \"\") for b in data.get(\"content\", []))\n\n\ndef make_handler(router):\n    class H(BaseHTTPRequestHandler):\n        def _send(self, code, obj):\n            b = json.dumps(obj).encode()\n            self.send_response(code); self.send_header(\"content-type\", \"application/json\")\n            self.send_header(\"content-length\", str(len(b))); self.end_headers(); self.wfile.write(b)\n\n        def log_message(self, *a):\n            pass\n\n        def do_GET(self):\n            self._send(200, {\"ok\": True, \"policy\": asdict(router.policy)}) if self.path == \"/health\" else self._send(404, {\"error\": \"not found\"})\n\n        def do_POST(self):\n            try:\n                body = json.loads(self.rfile.read(int(self.headers.get(\"content-length\", 0))) or b\"{}\")\n            except Exception:\n                return self._send(400, {\"error\": \"invalid JSON body\"})\n            if self.path == \"/v1/chat/completions\":\n                if not [m for m in body.get(\"messages\", []) if m.get(\"role\") == \"user\"]:\n                    return self._send(400, {\"error\": \"send at least one user message\"})\n                request, system, assistant, existing, full_user = parse_messages(body[\"messages\"])\n                res = router.run(request, system, assistant, existing, full_user)\n                if res[\"verified\"]:\n                    content = res[\"flow\"]\n                elif assistant:   # Studio reads steps/notes: an empty answer with the honest reason is shown to the user\n                    content = json.dumps({\"variables\": [], \"steps\": [], \"notes\": \"No flow was returned: \" + res[\"reason\"] + \". \" + \" \".join(res.get(\"problems\", [])[:2])})\n                else:\n                    content = json.dumps({\"error\": res[\"reason\"], \"problems\": res.get(\"problems\", [])})\n                pt = sum(len(str(m.get(\"content\", \"\"))) for m in body[\"messages\"]) // 4\n                return self._send(200, {\"id\": \"ooa-\" + str(int(time.time() * 1000)), \"object\": \"chat.completion\",\n                                        \"created\": int(time.time()), \"model\": body.get(\"model\") or \"ooa-assistant\",\n                                        \"choices\": [{\"index\": 0, \"finish_reason\": \"stop\" if res[\"verified\"] else \"length\",\n                                                     \"message\": {\"role\": \"assistant\", \"content\": content}}],\n                                        \"usage\": {\"prompt_tokens\": pt, \"completion_tokens\": len(content) // 4, \"total_tokens\": pt + len(content) // 4},\n                                        \"ooa\": {k: v for k, v in res.items() if k != \"flow\"}})\n            if self.path == \"/v1/feedback\":\n                return self._send(200, router.feedback(body))\n            self._send(404, {\"error\": \"not found\"})\n    return H\n\n\ndef serve(router, port=8080, host=\"127.0.0.1\"):\n    srv = ThreadingHTTPServer((host, port), make_handler(router))\n    return srv\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--catalog\", required=True)\n    ap.add_argument(\"--platform\", required=True)\n    ap.add_argument(\"--policy\", default=None)\n    ap.add_argument(\"--local\", default=None, help=\"'nanochat' to load the trained model (needs GPU/CPU + nanochat)\")\n    ap.add_argument(\"--source\", default=\"sft\")\n    ap.add_argument(\"--port\", type=int, default=8080)\n    a = ap.parse_args()\n    pol = Policy.load(a.policy)\n    local = None\n    if a.local == \"nanochat\":\n        from backends import NanochatBackend\n        local = NanochatBackend(a.source)\n    cloud = CloudBackend(pol.cloud_model, pol.cloud_max_tokens) if pol.allow_escalation else None\n    r = Router(a.catalog, a.platform, pol, local, cloud)\n    print(f\"OOA assistant on http://127.0.0.1:{a.port}  | local model: {bool(local)} | cloud escalation: {pol.allow_escalation}\")\n    serve(r, a.port).serve_forever()\n", "serving/backends.py": "\"\"\"Local model backend for the server. NanochatBackend loads our trained nanochat checkpoint (same calls as\ntraining/ooa_self_improve.py). NOT tested in the build environment (needs nanochat + torch); tested path uses fakes.\"\"\"\nimport sys\n\n\nclass NanochatBackend:\n    def __init__(self, source=\"sft\", max_new_tokens=1024, base_temp=0.2):\n        import torch\n        from nanochat.common import compute_init, autodetect_device_type\n        from nanochat.checkpoint_manager import load_model\n        from nanochat.engine import Engine\n        _, _, _, _, self.device = compute_init(autodetect_device_type())\n        self.model, self.tok, _ = load_model(source, self.device, phase=\"eval\")\n        self.engine = Engine(self.model, self.tok)\n        self.max_new_tokens, self.base_temp = max_new_tokens, base_temp\n\n    def generate(self, request, attempt=0):\n        conv = {\"messages\": [{\"role\": \"user\", \"content\": request}, {\"role\": \"assistant\", \"content\": \"\"}]}\n        tokens = self.tok.render_for_completion(conv)\n        seqs, _ = self.engine.generate_batch(tokens, num_samples=1, max_tokens=self.max_new_tokens,\n                                             temperature=self.base_temp + 0.3 * attempt, top_k=50, seed=1000 + attempt)\n        return self.tok.decode(seqs[0][len(tokens):])\n", "flow_check.py": "\"\"\"Flow scorer: JSON-schema validity + every action/input exists in the catalog.\nUsage: python flow_check.py PLATFORM_REPO catalog.jsonl\nWhy not only `ooa validate`: the Python reference lacks most toolkits (browser, desktop...), so it rejects valid flows.\n\"\"\"\nimport json,glob,sys\nimport jsonschema\nP=sys.argv[1]; C=sys.argv[2]\nschema=json.load(open(P+'/contracts/flow-schema/flow.v1.json'))\ncat={}\nfor l in open(C):\n    e=json.loads(l); cat[e['id']]=e\ndef steps(n):\n    for s in n:\n        yield s\n        for k in ('children','else'):\n            v=s.get(k)\n            if isinstance(v,list): yield from steps(v)\n        for ei in s.get('elseIf',[]) or []:\n            yield from steps(ei.get('children',[]))\n        for c in s.get('catch',[]) or []:\n            yield from steps(c.get('children',[]))\n        if isinstance(s.get('finally'),list): yield from steps(s['finally'])\ntot=sok=0; unk=bad_in=0; unk_ids=set(); ex=[]\nfor f in sorted(glob.glob(P+'/examples/**/*.flow.json',recursive=True)):\n    d=json.load(open(f)); tot+=1\n    try: jsonschema.validate(d,schema); sok+=1\n    except Exception as e: ex.append((f.split('/')[-1],str(e)[:100]))\n    for s in steps(d.get('steps',[])):\n        a=s.get('action') or s.get('use') or s.get('type')\n        if not a or not isinstance(a,str) or '/' not in a: continue\n        if a not in cat: unk+=1; unk_ids.add(a); continue\n        names={i['name'] for i in cat[a]['inputs']}\n        for k in (s.get('args') or s.get('inputs') or {}):\n            if k not in names: bad_in+=1\nprint('flows',tot,'schema_ok',sok,'unknown actions',unk,sorted(unk_ids)[:8],'unknown inputs',bad_in); print(ex[:3])\n", "plain_requests.jsonl": "{\"prompt\": \"Read a CSV file into a table, then write the table to an Excel sheet\", \"must\": [\"ooa.csv/read\", \"ooa.excel/write_table\"]}\n{\"prompt\": \"Open the website and save its product table as a CSV file\", \"must\": [\"ooa.browser/open\", \"ooa.browser/extract_table\", \"ooa.csv/write\"]}\n{\"prompt\": \"Copy a file to another folder, then delete the original\", \"must\": [\"ooa.file/copy\", \"ooa.file/delete\"]}\n{\"prompt\": \"Get the text out of a PDF and save it to a text file\", \"must\": [\"ooa.pdf/extract_text\", \"ooa.file/write_text\"]}\n{\"prompt\": \"Merge some PDF files into one\", \"must\": [\"ooa.pdf/merge\"]}\n{\"prompt\": \"Open a workbook, read the whole sheet as a table, and close it\", \"must\": [\"ooa.excel/open\", \"ooa.excel/get_all_cells\", \"ooa.excel/close\"]}\n{\"prompt\": \"Read a CSV, sort it, and write it back to a CSV\", \"must\": [\"ooa.csv/read\", \"ooa.datatable/sort\", \"ooa.csv/write\"]}\n{\"prompt\": \"Read a CSV and count its rows\", \"must\": [\"ooa.csv/read\", \"ooa.datatable/row_count\"]}\n{\"prompt\": \"Log in to a website: open the browser, type the username and password, click Sign in\", \"must\": [\"ooa.browser/open\", \"ooa.browser/type_text\", \"ooa.browser/type_text\", \"ooa.browser/click\"]}\n{\"prompt\": \"Read a text file and write its contents to a new file\", \"must\": [\"ooa.file/read_text\", \"ooa.file/write_text\"]}\n{\"prompt\": \"Open a Word document, replace some text, and save it\", \"must\": [\"ooa.word/open\", \"ooa.word/replace_text\", \"ooa.word/save\"]}\n{\"prompt\": \"For each file in a list, move it to the archive folder\", \"must\": [\"core/loop\", \"ooa.file/move\"]}\n{\"prompt\": \"Count the pages of a PDF\", \"must\": [\"ooa.pdf/page_count\"]}\n{\"prompt\": \"Pull the tables out of a PDF and write the first one to Excel\", \"must\": [\"ooa.pdf/extract_tables\", \"ooa.excel/write_table\"]}\n{\"prompt\": \"Open a web page, read the text of the price, then close the browser\", \"must\": [\"ooa.browser/open\", \"ooa.browser/get_text\", \"ooa.browser/close\"]}\n{\"prompt\": \"Read a CSV of orders, keep only the unpaid ones, and save them to a new CSV\", \"must\": [\"ooa.csv/read\", \"ooa.datatable/filter\", \"ooa.csv/write\"]}\n{\"prompt\": \"Save the attachments of an email to a folder\", \"must\": [\"ooa.email/save_attachments\"]}\n{\"prompt\": \"List the files in a folder\", \"must\": [\"ooa.file/list\"]}\n{\"prompt\": \"Download a file from a web page\", \"must\": [\"ooa.browser/download\"]}\n{\"prompt\": \"Create a Word document and add a paragraph to it, then save it\", \"must\": [\"ooa.word/create_document\", \"ooa.word/add_paragraph\", \"ooa.word/save\"]}\n"}
for name, text in FILES.items():
    os.makedirs(os.path.dirname(f"/kaggle/working/ooa/{name}"), exist_ok=True)
    open(f"/kaggle/working/ooa/{name}", "w", encoding="utf-8").write(text)
if not os.path.exists("/kaggle/working/platform"):
    !git clone -q --depth 1 {PLATFORM_REPO} /kaggle/working/platform
!python /kaggle/working/ooa/build_sft_data.py /kaggle/working/platform {DRIVE_DIR}/sft {MAX_FLOW_CHARS}
!python /kaggle/working/ooa/gen_flows.py {DRIVE_DIR}/sft/catalog.jsonl /kaggle/working/platform {DRIVE_DIR}/sft/synth_train.jsonl {SYNTH_TRAIN} 1
!python /kaggle/working/ooa/gen_flows.py {DRIVE_DIR}/sft/catalog.jsonl /kaggle/working/platform /kaggle/working/synth_eval_raw.jsonl 400 2
train_set = {json.loads(l)["messages"][1]["content"] for l in open(f"{DRIVE_DIR}/sft/synth_train.jsonl")}
def _all_actions(nodes):   # every step, including those inside if / try / loop (the test compares all of them)
    out = []
    for n in nodes or []:
        out.append(n.get("action"))
        for k in ("children", "else", "finally"):
            if isinstance(n.get(k), list): out += _all_actions(n[k])
        for ei in n.get("elseIf") or []: out += _all_actions(ei.get("children"))
        for c in n.get("catch") or []: out += _all_actions(c.get("children"))
    return out
ev = []
for l in open("/kaggle/working/synth_eval_raw.jsonl"):
    m = json.loads(l)["messages"]
    if m[1]["content"] not in train_set and len(ev) < SYNTH_EVAL:
        ev.append({"prompt": m[0]["content"], "gold_actions": _all_actions(json.loads(m[1]["content"])["steps"])})
open(f"{DRIVE_DIR}/sft/eval_synth.jsonl", "w").write("\n".join(json.dumps(x) for x in ev))
# validation set the model never trains on (the old one overlapped training, so its loss looked great for nothing)
val_rows = [json.loads(l) for l in open(f"{DRIVE_DIR}/sft/val.jsonl")]
val_rows = [r for r in val_rows if r["messages"][1]["content"].startswith("Use ") or r["messages"][1]["content"].startswith("`")]
raw = [json.loads(l) for l in open("/kaggle/working/synth_eval_raw.jsonl")]
extra = [r for r in raw[SYNTH_EVAL:] if r["messages"][1]["content"] not in train_set][:150]
with open(f"{DRIVE_DIR}/sft/val.jsonl", "w") as f:
    for r in val_rows + extra:
        f.write(json.dumps(r) + "\n")
print("validation rows (unseen):", len(val_rows) + len(extra))
print("synthetic eval prompts (not in train):", len(ev))
!python /kaggle/working/ooa/gen_flows.py {DRIVE_DIR}/sft/catalog.jsonl /kaggle/working/platform {DRIVE_DIR}/sft/synth_pattern.jsonl {SYNTH_PATTERN} 1 patterns
!python /kaggle/working/ooa/gen_flows.py {DRIVE_DIR}/sft/catalog.jsonl /kaggle/working/platform /kaggle/working/pattern_eval_raw.jsonl 200 2 patterns
_ptrain = {json.loads(l)["messages"][1]["content"] for l in open(f"{DRIVE_DIR}/sft/synth_pattern.jsonl")}
_pev = []
for l in open("/kaggle/working/pattern_eval_raw.jsonl"):
    m = json.loads(l)["messages"]
    if m[1]["content"] not in _ptrain and len(_pev) < SYNTH_EVAL:
        _f = json.loads(m[1]["content"])
        _pev.append({"prompt": m[0]["content"], "gold_flow": m[1]["content"]})
open(f"{DRIVE_DIR}/sft/eval_pattern.jsonl", "w").write("\n".join(json.dumps(x) for x in _pev))
print("real-pattern eval prompts (not in train):", len(_pev))
!cp /kaggle/working/ooa/ooa_task.py /kaggle/working/nanochat/tasks/ooa.py
!head -c 700 {DRIVE_DIR}/sft/train.jsonl

In [ ]:
#@title 7. Patch nanochat SFT to use our data, then train
import re, shutil
# always start from the pinned nanochat scripts, so an earlier patch can never hide the current data mix
!cd /kaggle/working/nanochat && git checkout -q -- scripts/chat_sft.py scripts/chat_cli.py
p = "/kaggle/working/nanochat/scripts/chat_sft.py"
s = open(p).read()
if "OOADialogues" not in s:
    s = s.replace("from tasks.smoltalk import SmolTalk", "from tasks.smoltalk import SmolTalk\nfrom tasks.ooa import OOADialogues")
    a = s.index("train_tasks = [")
    b = s.index("\n", s.index("# total: 24K"))
    new = f"""train_tasks = ([OOADialogues("{DRIVE_DIR}/sft/train_flows.jsonl") for _ in range({FLOW_REPEAT})]
               + [OOADialogues("{DRIVE_DIR}/sft/train_catalog.jsonl"), OOADialogues("{DRIVE_DIR}/sft/synth_train.jsonl"),
                  OOADialogues("{DRIVE_DIR}/sft/synth_pattern.jsonl")]) * {SFT_EPOCHS}
import os as _os
_extra = [OOADialogues("{DRIVE_DIR}/sft/" + _f) for _f in ("self_improved.jsonl", "hard_fixed.jsonl", "feedback_train.jsonl")
          if _os.path.exists("{DRIVE_DIR}/sft/" + _f)]
train_tasks += _extra * {SFT_EPOCHS}
print0(f"extra self-learning files in the mix: {{len(_extra)}}")
if {SMOLTALK_ROWS} > 0:
    train_tasks.append(SmolTalk(split="train", stop={SMOLTALK_ROWS}))
train_dataset = TaskMixture(train_tasks)
print0(f"OOA training mixture: {{len(train_dataset):,}} rows")
val_dataset = TaskMixture([OOADialogues("{DRIVE_DIR}/sft/val.jsonl")])"""
    s = s[:a] + new + s[b:]
    # never train on a cut-off flow: nanochat truncates conversations to 2048 tokens; skip ones longer than a row instead
    old_r = "            ids, mask = tokenizer.render_conversation(conversation)\n            conv_buffer.append((ids, mask))"
    new_r = ("            ids, mask = tokenizer.render_conversation(conversation, max_tokens=10**9)\n"
             "            if len(ids) <= row_capacity:\n"
             "                conv_buffer.append((ids, mask))\n"
             "            else:\n"
             "                skipped_long[0] += 1")
    assert old_r in s, "chat_sft.py loader changed upstream"
    s = s.replace(old_r, new_r)
    s = s.replace("last_step = False # we will toggle this", "skipped_long = [0]\nlast_step = False # we will toggle this")
    s = s.replace('print0(f"Total training time:', 'print0(f"Conversations skipped as longer than the window (not cut): {skipped_long[0]}")\nprint0(f"Total training time:')
    open(p, "w").write(s)
    print("patched chat_sft.py")
# remove old SFT checkpoints: nanochat loads the highest step number, so a stale step-100 would hide a newer, shorter run
import shutil
shutil.rmtree(os.path.join(os.environ["NANOCHAT_BASE_DIR"], "chatsft_checkpoints"), ignore_errors=True)
!python -m scripts.chat_sft --max-seq-len={SFT_SEQ} --device-batch-size={SFT_DEVICE_BATCH} \
    --total-batch-size={SFT_SEQ*SFT_DEVICE_BATCH*4} --num-iterations={SFT_ITERS} \
    --eval-every=100 --eval-tokens={SFT_SEQ*SFT_DEVICE_BATCH*8} --chatcore-every=-1 --run=dummy

In [ ]:
#@title 8. Test on held-out workflow families (never trained on)
import subprocess, json, sys
EVAL_SOURCE = globals().get("EVAL_SOURCE", "sft")
print("evaluating model source:", EVAL_SOURCE)
# chat_cli stops after 256 new tokens; flows are longer, so raise the limit
cli = "/kaggle/working/nanochat/scripts/chat_cli.py"
t = open(cli).read()
t = t.replace('"max_tokens": 256', '"max_tokens": 3000')
open(cli, "w").write(t)
sys.path.insert(0, "/kaggle/working/ooa")
import jsonschema
schema = json.load(open("/kaggle/working/platform/contracts/flow-schema/flow.v1.json"))
catalog = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DRIVE_DIR}/sft/catalog.jsonl")}

def steps(nodes):
    for s in nodes:
        yield s
        for k in ("children", "else"):
            if isinstance(s.get(k), list): yield from steps(s[k])
        for ei in s.get("elseIf") or []: yield from steps(ei.get("children", []))
        for c in s.get("catch") or []: yield from steps(c.get("children", []))
        if isinstance(s.get("finally"), list): yield from steps(s["finally"])

results = []
for row in map(json.loads, open(f"{DRIVE_DIR}/sft/eval.jsonl")):
    out = subprocess.run(["python", "-m", "scripts.chat_cli", "-i", EVAL_SOURCE, "-p", row["prompt"], "-t", "0.2"],
                         capture_output=True, text=True, cwd="/kaggle/working/nanochat").stdout
    text = out.split("Assistant:", 1)[-1].split("<|assistant_end|>")[0].strip()   # drop the end-of-answer marker
    print("---", row["family"], "| first 250 chars:", text[:250])
    r = {"family": row["family"], "json": False, "schema": False, "actions_known": False}
    try:
        flow = json.loads(text); r["json"] = True
        jsonschema.validate(flow, schema); r["schema"] = True
        acts = [s.get("action") for s in steps(flow.get("steps", [])) if isinstance(s.get("action"), str) and "/" in s["action"]]
        r["actions_known"] = all(a in catalog for a in acts)
    except Exception as e:
        r["error"] = str(e)[:80]
    results.append(r); print(r)
n = len(results)
print(f"\nheld-out flows: {n} | valid JSON {sum(r['json'] for r in results)} | schema OK {sum(r['schema'] for r in results)} | all actions exist {sum(r['actions_known'] for r in results)}")
print("\n=== Synthetic eval: unseen generated flows, exact action sequence ===")
sr = []
for row in map(json.loads, open(f"{DRIVE_DIR}/sft/eval_synth.jsonl")):
    out = subprocess.run(["python", "-m", "scripts.chat_cli", "-i", EVAL_SOURCE, "-p", row["prompt"], "-t", "0.2"],
                         capture_output=True, text=True, cwd="/kaggle/working/nanochat").stdout
    text = out.split("Assistant:", 1)[-1].split("<|assistant_end|>")[0].strip()
    r = {"json": False, "schema": False, "actions_known": False, "actions_match": False}
    try:
        flow = json.loads(text); r["json"] = True
        jsonschema.validate(flow, schema); r["schema"] = True
        acts = [x.get("action") for x in steps(flow.get("steps", []))]
        r["actions_known"] = all(a in catalog for a in acts)
        r["actions_match"] = acts == row["gold_actions"]
    except Exception:
        pass
    sr.append(r)
print(f"synthetic eval: {len(sr)} | valid JSON {sum(r['json'] for r in sr)} | schema OK {sum(r['schema'] for r in sr)} | actions exist {sum(r['actions_known'] for r in sr)} | exact action sequence {sum(r['actions_match'] for r in sr)}")
print("\n=== Real-pattern eval: unseen long, multi-toolkit flows (closest thing to real flows) ===")
sys.path.insert(0, "/kaggle/working/ooa")
import ooa_reward as RW
os.environ["OOA_CATALOG"] = f"{DRIVE_DIR}/sft/catalog.jsonl"; os.environ["OOA_SCHEMA"] = "/kaggle/working/platform/contracts/flow-schema/flow.v1.json"
RW.configure()
pr = []
for row in map(json.loads, open(f"{DRIVE_DIR}/sft/eval_pattern.jsonl")):
    out = subprocess.run(["python", "-m", "scripts.chat_cli", "-i", EVAL_SOURCE, "-p", row["prompt"], "-t", "0.2"],
                         capture_output=True, text=True, cwd="/kaggle/working/nanochat").stdout
    sc = RW.score(out.split("Assistant:", 1)[-1], RW.gold_actions(row["gold_flow"]))
    pr.append(sc)
print(f"real-pattern eval: {len(pr)} | valid (passes every rule) {sum(x['valid'] for x in pr)} | exact action sequence {sum(x['solved'] for x in pr)} | mean score {sum(x['total'] for x in pr)/max(1,len(pr)):.3f}")
print("\n=== Plain-wording eval: 20 short requests written the way people talk (hand-written, never generated) ===")
def _in_order(need, got):
    it = iter(got)
    return all(any(a == g for g in it) for a in need)
pl = []
for row in map(json.loads, open("/kaggle/working/ooa/plain_requests.jsonl")):
    out = subprocess.run(["python", "-m", "scripts.chat_cli", "-i", EVAL_SOURCE, "-p", row["prompt"], "-t", "0.2"],
                         capture_output=True, text=True, cwd="/kaggle/working/nanochat").stdout
    sc = RW.score(out.split("Assistant:", 1)[-1])
    try: got = RW.actions_of(json.loads(RW.clean(out.split("Assistant:", 1)[-1])))
    except Exception: got = []
    pl.append((sc["valid"], sc["valid"] and _in_order(row["must"], got)))
    print(("OK  " if pl[-1][1] else ("VALID" if pl[-1][0] else "FAIL ")), row["prompt"][:70], "->", got[:6])
print(f"plain-wording eval: {len(pl)} | valid (passes every rule) {sum(v for v, _ in pl)} | has the needed actions in order {sum(o for _, o in pl)}")
print("\nTry it yourself:")
!cd /kaggle/working/nanochat && python -m scripts.chat_cli -i sft -p "Which OOA action would I use to read a CSV file into a table?" -t 0.2

## Reinforcement layer: the model learns from the platform's own rules

Two loops, both scored by a **verifier** (`ooa_reward.py`: schema, actions exist, required inputs, declared variables,
action sequence vs the known-good flow). No human labels and no LLM judge are needed.

1. **RL (cell 9):** nanochat's `chat_rl.py` writes 8 answers per request; answers the verifier scores above the group
   average are reinforced. Partial credit gives a gradient even when no answer is perfect.
2. **Self-improvement rounds (cell 10):** each round the model attempts *fresh* requests. Verified successes are kept as new
   training data; requests it fails on are kept with the verified correct flow and retried next round. Each round then
   retrains from the pretrained base on all data so far (never only on its own output, which drifts). The pass rate on
   fresh requests per round is the progress measure.

Run cell 7 (SFT) once before these. Real user feedback joins later through `ingest_feedback.py`.

In [ ]:
#@title 9. Reinforcement learning with the verifier as reward
import json, os, shutil, sys
os.environ["OOA_CATALOG"] = f"{DRIVE_DIR}/sft/catalog.jsonl"
os.environ["OOA_SCHEMA"] = "/kaggle/working/platform/contracts/flow-schema/flow.v1.json"
sys.path.insert(0, "/kaggle/working/ooa")
import gen_flows as G
catalog = [json.loads(l) for l in open(os.environ["OOA_CATALOG"], encoding="utf-8")]
used = set()
for f in ("synth_train.jsonl", "self_improved.jsonl", "hard_fixed.jsonl"):
    if os.path.exists(f"{DRIVE_DIR}/sft/{f}"):
        used |= {json.loads(l)["messages"][1]["content"] for l in open(f"{DRIVE_DIR}/sft/{f}")}
def rows(n, seed, exclude):
    out = G.generate(catalog, "/kaggle/working/platform", n, seed, exclude=exclude)
    out.sort(key=lambda t: len(t[1]["steps"]))      # short (easy) requests first: a simple curriculum
    return out
tr = rows(RL_PROMPTS, 3, used)
va = rows(40, 4, used | {m for _, _, m in tr})
for name, data in (("rl_train", tr), ("rl_val", va)):
    with open(f"{DRIVE_DIR}/sft/{name}.jsonl", "w", encoding="utf-8") as f:
        for q, _, mini in data:
            f.write(json.dumps({"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": mini}]}) + "\n")
print("RL requests:", len(tr), "train /", len(va), "val")

!cp /kaggle/working/ooa/ooa_rl_task.py /kaggle/working/nanochat/tasks/ooa_rl.py
!cd /kaggle/working/nanochat && git checkout -q -- scripts/chat_rl.py
p = "/kaggle/working/nanochat/scripts/chat_rl.py"
s = open(p).read()
for old, new in [
    ("from tasks.gsm8k import GSM8K", "from tasks.ooa_rl import OOAFlowRL"),
    ('train_task = GSM8K(subset="main", split="train")', f'train_task = OOAFlowRL("{DRIVE_DIR}/sft/rl_train.jsonl")'),
    ('val_task = GSM8K(subset="main", split="test")', f'val_task = OOAFlowRL("{DRIVE_DIR}/sft/rl_val.jsonl")'),
    ("max_completion_tokens=256,", "max_completion_tokens=1024,"),
]:
    assert old in s, "chat_rl.py changed upstream, cannot patch: " + old
    s = s.replace(old, new)
open(p, "w").write(s)
shutil.rmtree(os.path.join(os.environ["NANOCHAT_BASE_DIR"], "chatrl_checkpoints"), ignore_errors=True)
!cd /kaggle/working/nanochat && python -m scripts.chat_rl --num-samples={RL_SAMPLES} --device-batch-size={RL_SAMPLES} \
    --examples-per-step={RL_EXAMPLES_PER_STEP} --max-new-tokens=1024 --eval-every={RL_EVERY} --eval-examples=40 \
    --save-every={RL_EVERY} --num-epochs=1 --run=dummy

In [ ]:
#@title 9b. Point the test at the RL model, then re-run cell 8
EVAL_SOURCE = "rl"     # cell 8 prints "evaluating model source: rl". Set back to "sft" to test the SFT model.

In [ ]:
#@title 10. Self-improvement rounds: attempt fresh requests, keep verified wins, learn from failures, retrain
import json, os, shutil
os.environ["OOA_CATALOG"] = f"{DRIVE_DIR}/sft/catalog.jsonl"
os.environ["OOA_SCHEMA"] = "/kaggle/working/platform/contracts/flow-schema/flow.v1.json"
!cp /kaggle/working/ooa/ooa_self_improve.py /kaggle/working/nanochat/scripts/ooa_self_improve.py
# guards: cell 10 reuses the data and the patched chat_sft.py from cells 6-7. Stop if they do not match the config.
import re as _re
_n = sum(1 for _ in open(f"{DRIVE_DIR}/sft/synth_train.jsonl"))
_patched = open("/kaggle/working/nanochat/scripts/chat_sft.py").read()
_ep = _re.search(r"\) \* (\d+)\n", _patched)
print(f"synth_train.jsonl rows: {_n} (config SYNTH_TRAIN={SYNTH_TRAIN}) | patched SFT epochs: {_ep.group(1) if _ep else '?'} (config {SFT_EPOCHS})")
if _n < 0.9 * SYNTH_TRAIN or (_ep and int(_ep.group(1)) != SFT_EPOCHS):
    raise RuntimeError("Data or SFT patch is out of date for the current config. Run cell 1, cell 6 (data), cell 7 (SFT), then run this cell again.")
sft_cmd = (f"python -m scripts.chat_sft --max-seq-len={SFT_SEQ} --device-batch-size={SFT_DEVICE_BATCH} "
           f"--total-batch-size={SFT_SEQ*SFT_DEVICE_BATCH*4} --num-iterations={SFT_ITERS} "
           f"--eval-every=100 --eval-tokens={SFT_SEQ*SFT_DEVICE_BATCH*8} --chatcore-every=-1 --run=dummy")
for r in range(SELF_IMPROVE_ROUNDS):
    print(f"\n===== self-improvement round {r + 1}/{SELF_IMPROVE_ROUNDS} =====")
    !cd /kaggle/working/nanochat && python -m scripts.ooa_self_improve --source sft --out {DRIVE_DIR}/sft --seed {100 + r} --num-prompts {SI_PROMPTS} --samples {SI_SAMPLES}
    shutil.rmtree(os.path.join(os.environ["NANOCHAT_BASE_DIR"], "chatsft_checkpoints"), ignore_errors=True)
    !cd /kaggle/working/nanochat && {sft_cmd}
print("\nPass rate on FRESH requests, per round (this is the progress measure):")
for l in open(f"{DRIVE_DIR}/sft/selfimprove_log.jsonl"):
    x = json.loads(l)
    print(x["time"], "| pass@k", x["fresh_pass_at_k"], "| mean reward", x["mean_reward"], "| new wins", x["added_self_improved"], "| new fixes", x["added_hard_fixed"], "| still hard", x["hard_queue"])
    print("   where answers lose points (1.0 = always fine):", x.get("parts"))

## Try the whole assistant (local only)

Runs the same router the server uses: our model writes the flow, typo-level repair is applied, and the platform's rules verify it.
Cloud escalation is **off** here (no key needed), so this shows what our own model can do alone. It also runs the server's
local-model code on a GPU for the first time.

In [ ]:
#@title 13. The whole assistant, local only: model -> repair -> verify
import json, os, sys
sys.path[:0] = ["/kaggle/working/ooa", "/kaggle/working/ooa/serving", "/kaggle/working/nanochat"]
os.chdir("/kaggle/working/nanochat")
import ooa_server as S
import ooa_reward as R
from backends import NanochatBackend

policy = S.Policy(allow_escalation=False, max_clauses_local=99, local_retries=2)    # local only; do not skip the local model
local = NanochatBackend("sft", max_new_tokens=1024)
router = S.Router(f"{DRIVE_DIR}/sft/catalog.jsonl", "/kaggle/working/platform", policy, local, None)
rows = [json.loads(l) for l in open(f"{DRIVE_DIR}/sft/eval_synth.jsonl")]
verified = exact = repaired = 0
for r in rows:
    res = router.run(r["prompt"])
    if res["verified"]:
        verified += 1
        repaired += bool(res["repairs"])
        got = R.actions_of(json.loads(res["flow"]))
        exact += got == r["gold_actions"]
    print(("OK   " if res["verified"] else "FAIL ") + f"{res['seconds']:>5}s attempts={len(res['attempts'])} repairs={len(res.get('repairs', []))}"
          + ("" if res["verified"] else " | " + "; ".join(res.get("problems", [])[:2])[:110]))
print(f"\nverified (passed every rule): {verified}/{len(rows)} | fixed by repair: {repaired} | exact right actions: {exact}/{len(rows)}")

## Autoresearch: the training recipe improves itself

Idea from [karpathy/autoresearch](https://github.com/karpathy/autoresearch) (MIT): fixed budget, one honest metric, change one
thing, keep it only if it helps. Here each experiment re-trains the SFT stage for a fixed **3M tokens (92 optimizer steps)** with one change to
the recipe (data mix or learning settings), then scores it on a **frozen** test of 40 generated + 4 real flows using our
verifier. Better than the best by more than 0.003 -> kept; otherwise discarded. Resumable: results are saved to Drive/working dir.

Each experiment takes about 10 minutes on a T4 (my estimate: 6-8 minutes training + 3 minutes scoring; not yet measured).
Run it after SFT works (cells 6-8). With an AI agent instead of this scripted search, see `autoresearch/program.md`.

In [ ]:
#@title 12. Autoresearch: scripted search over the SFT recipe (fixed 3M-token budget, frozen scoreboard)
AR_EXPERIMENTS = 6          # experiments this session (including the baseline); about 10 min each
import base64, json, os, subprocess, sys, zlib
AR = "/kaggle/working/ooa/autoresearch"
os.makedirs(AR + "/frozen", exist_ok=True)
_blob = "..."
for _n, _t in json.loads(zlib.decompress(base64.b64decode(_blob))).items():
    open(f"{AR}/frozen/{_n}", "w", encoding="utf-8").write(_t)
sys.path.insert(0, AR)
import loop
from recipe import RECIPE
work = f"{DRIVE_DIR}/autoresearch"
os.makedirs(work, exist_ok=True)

def runner(recipe):
    open(f"{work}/recipe_try.py", "w").write("from constants import FIXED_TOKEN_BUDGET\nRECIPE = " + repr(recipe) + "\n")
    cmd = [sys.executable, f"{AR}/run_experiment.py", "--platform", "/kaggle/working/platform", "--catalog", f"{DRIVE_DIR}/sft/catalog.jsonl",
           "--workdir", f"{work}/run", "--nanochat", "/kaggle/working/nanochat", "--base-dir", os.environ["NANOCHAT_BASE_DIR"],
           "--recipe", f"{work}/recipe_try.py", "--ooa-dir", "/kaggle/working/ooa"]
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-600:])
    if p.returncode != 0:
        raise RuntimeError((p.stdout + p.stderr)[-300:])
    return json.load(open(f"{work}/run/result.json"))

import shutil
live = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "chatsft_checkpoints")
def save_best(recipe, score):                       # the model just trained is the best so far: keep a copy on Drive
    shutil.rmtree(f"{work}/best_model", ignore_errors=True)
    shutil.copytree(live, f"{work}/best_model")
    json.dump({"score": score, "recipe": recipe}, open(f"{work}/best_model/best_recipe.json", "w"))
_rows = loop.read_results(f"{work}/results.tsv")
if _rows and not os.path.exists(f"{work}/best_model"):
    # an earlier session found a best recipe but its model was not saved: train it once more (also measures run-to-run noise)
    _top = [r for r in _rows if r["status"] in ("baseline", "keep")][-1]
    print(f"Rebuilding the best model: re-training the best recipe once (logged score {_top['score']})")
    _res = runner(json.loads(_top["recipe_json"]))
    loop.append_result(f"{work}/results.tsv", {"n": len(_rows), "score": f"{_res['score']:.4f}", "real_score": _res.get("real_score", ""),
        "solved_rate": _res.get("solved_rate", ""), "val_bpb": _res.get("val_bpb", ""), "minutes": _res.get("minutes", ""),
        "status": "recheck", "description": f"re-train of best (logged {_top['score']}): run-to-run noise = {_res['score'] - float(_top['score']):+.4f}",
        "recipe_json": _top["recipe_json"]})
    save_best(json.loads(_top["recipe_json"]), _res["score"])
    print(f"re-trained score {_res['score']:.4f} vs logged {_top['score']}  (difference = run-to-run noise)")
best, best_score = loop.run_loop(runner, RECIPE, f"{work}/results.tsv", max_experiments=AR_EXPERIMENTS, on_keep=save_best)
if os.path.exists(f"{work}/best_model"):             # leave the BEST model (not the last experiment's) as the live SFT model
    shutil.rmtree(live, ignore_errors=True)
    shutil.copytree(f"{work}/best_model", live)
    print("live SFT model = best experiment (score", best_score, "). Your earlier model is in chatsft_checkpoints_before_autoresearch.")
print("\nBest recipe so far:", json.dumps(best))
print("\nAll experiments:")
for r in loop.read_results(f"{work}/results.tsv"):
    print(f"#{r['n']:>2} {r['status']:8s} score {r['score']:>7} real {r['real_score']:>7} | {r['description']}")

## After this run
- Checkpoints: `DRIVE_DIR/nanochat_base/base_checkpoints` (pretrained) and `chatsft_checkpoints` (OOA-trained).
- Weak results on the held-out test are expected with the seed data. The next lever is more and better dialogues
  (edits, error fixing, missing-input questions), validated with the scorer in `eval/flow_check.py`.
- Reinforcement layer: cells 9-10 (RL with the verifier as reward; self-improvement rounds). Feed real usage back with `ingest_feedback.py`.
- For a stronger base, use `PROFILE = "t4"` and let it run (resume after disconnects), or an A100 on Colab Pro.

In [ ]:
#@title 11. Package the trained model (download it from the notebook's Output tab after "Save Version")
import glob, os, re, shutil
base = os.environ["NANOCHAT_BASE_DIR"]
out = "/kaggle/working/ooa_model_bundle"
shutil.rmtree(out, ignore_errors=True)
for src in ("base_checkpoints", "chatsft_checkpoints", "chatrl_checkpoints"):
    d = f"{base}/{src}/d{P['depth']}"
    steps = sorted(glob.glob(d + "/model_*.pt"))
    if not steps:
        continue
    tag = re.search(r"model_(\d+)\.pt", steps[-1]).group(1)
    dst = f"{out}/nanochat_base/{src}/d{P['depth']}"
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(d + f"/*_{tag}*"):          # model, meta and optimizer files of the latest step
        shutil.copy(f, dst)
    print("packed", src, "step", tag)
shutil.copytree(f"{base}/tokenizer", f"{out}/nanochat_base/tokenizer")
shutil.make_archive("/kaggle/working/ooa_model_bundle", "zip", out)
print("bundle:", round(os.path.getsize("/kaggle/working/ooa_model_bundle.zip") / 1e6), "MB at /kaggle/working/ooa_model_bundle.zip")